# ควอนตัมคอมพิวติ้งจากศูนย์ — ฉบับลงมือทำ

**สอนด้วยโค้ดล้วน ๆ ทีละขั้น** ไม่ต้องมีพื้นฐานควอนตัมมาก่อน ขอแค่เคยเขียน Python นิดหน่อย

---

### หน้านี้ต่างจากที่อื่นยังไง

ส่วนใหญ่ของ notebook นี้ **ไม่ใช้ Qiskit** แต่สร้างเครื่องจำลองควอนตัมขึ้นมาเองด้วย `numpy` ทีละบรรทัด
เหตุผลคือ Qiskit ซ่อนคณิตศาสตร์ไว้หมด พอเรียกฟังก์ชัน `qc.h(0)` แล้วได้ผลลัพธ์มา เราจะไม่มีทางรู้เลยว่าข้างในเกิดอะไรขึ้น

พอเราเขียนเองแล้วจะเห็นชัดว่า **ควอนตัมคอมพิวติ้ง = พีชคณิตเชิงเส้น + กฎสุ่มหนึ่งข้อ** เท่านั้นจริง ๆ
ตอนท้าย (ตอนที่ 19) ค่อยกลับมาใช้ Qiskit เพื่อเทียบว่าได้ผลตรงกัน

### วิธีใช้

1. **รันเซลล์ตั้งค่า (ตอนที่ 0) ก่อนเสมอ** แล้วค่อยไล่ลงมาตามลำดับ
2. ทุกตอนจบด้วย **แบบฝึกหัด** — ลองเขียนเองก่อน แล้วค่อยกดเปิดเซลล์เฉลยที่อยู่ถัดลงมา
3. อย่าแค่กดรัน — **ลองแก้ตัวเลขแล้วรันซ้ำ** ดูว่าอะไรเปลี่ยน นั่นคือจุดที่ความเข้าใจเกิดขึ้น

### สารบัญ

| ตอน | เรื่อง | ตรงกับสไลด์หน้า |
|---|---|---|
| 0 | ตั้งค่า | — |
| 1 | เวกเตอร์และจำนวนเชิงซ้อนใน numpy | — |
| 2 | คิวบิตคืออะไร | 4, 8 |
| 3 | การวัดและกฎของบอร์น | 6–7 |
| 4 | ทรงกลมบลอค | 8–9 |
| 5 | ประตูควอนตัม 1 คิวบิต | 10–11 |
| 6 | วงจร = การคูณเมทริกซ์ | 10 |
| 7 | เมทริกซ์ความหนาแน่น | 4–5 |
| 8 | ผลคูณเทนเซอร์ · หลายคิวบิต | 12 |
| 9 | CNOT และสถานะเบลล์ทั้งสี่ | 13–16 |
| 10 | การพัวพัน · วัดด้วยอันดับเมทริกซ์ | 15 |
| 11 | เทเลพอร์ตสถานะควอนตัม | 17–19 |
| 12 | Superdense coding | 20–27 |
| 13 | วงจรบวกเลข (Toffoli) | 28–31 |
| 14 | Deutsch–Jozsa | 29–34 |
| 15 | อัลกอริทึมของโกรเวอร์ | 46–51 |
| 16 | QFT | 35–36 |
| 17 | Quantum Phase Estimation | 37–40 |
| 18 | อัลกอริทึมของชอร์ | 41–45 |
| 19 | ทำซ้ำด้วย Qiskit ของจริง | 28–31 |
| 20 | สรุปและทางไปต่อ | 52 |

> อ้างอิงเนื้อหาตามสไลด์ *Introduction to Quantum Computing* โดย Kamonluk Suksen, Ph.D.

---
# ตอนที่ 0 — ตั้งค่า

รันเซลล์นี้ครั้งเดียวก่อนเริ่ม มันจะโหลด `numpy` กับ `matplotlib` และเตรียมฟังก์ชันวาดกราฟที่เราจะใช้ตลอดทั้ง notebook

ยังไม่ต้องเข้าใจโค้ดในเซลล์นี้ทั้งหมด — เดี๋ยวเราจะค่อย ๆ สร้างฟังก์ชันพวกนี้ขึ้นมาเองในตอนถัด ๆ ไป

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True, linewidth=120)

# ---- จานสีที่ผ่านการตรวจเรื่องตาบอดสี ----
POS   = "#eb6834"   # แอมพลิจูดบวก
NEG   = "#2a78d6"   # แอมพลิจูดลบ
MARK  = "#4a3aa7"   # ตัวที่อยากเน้น
INK   = "#52514e"
MUTED = "#898781"
GRID  = "#e1e0d9"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": MUTED, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.titlesize": 11, "axes.labelsize": 10,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 110,
})

def labels_for(n):
    """คืนชื่อสถานะฐานทั้งหมดของระบบ n คิวบิต เช่น n=2 -> ['00','01','10','11']"""
    return [format(i, "0{}b".format(n)) for i in range(2 ** n)]

def nqubits(state):
    return int(round(np.log2(len(state))))

def plot_amplitudes(state, title="amplitudes", mark=None, ax=None):
    """วาดแท่งแอมพลิจูด: ขึ้นบน = บวก, ลงล่าง = ลบ (ทิศทางบอกเครื่องหมาย ไม่ได้พึ่งสีอย่างเดียว)"""
    state = np.asarray(state, dtype=complex)
    n = nqubits(state)
    if np.max(np.abs(state.imag)) > 1e-9:
        print("หมายเหตุ: สถานะนี้มีส่วนจินตภาพ กราฟแสดงเฉพาะส่วนจริง")
    vals = state.real
    if ax is None:
        _, ax = plt.subplots(figsize=(min(10, 1.0 + 0.45 * len(vals)), 2.9))
    colors = [POS if v >= 0 else NEG for v in vals]
    if mark is not None:
        mark = set(mark if hasattr(mark, "__iter__") else [mark])
        colors = [MARK if i in mark else c for i, c in enumerate(colors)]
    ax.bar(range(len(vals)), vals, color=colors, width=0.68)
    ax.axhline(0, color=MUTED, lw=1)
    ax.set_ylim(-1.08, 1.08)
    ax.set_ylabel("amplitude")
    ax.set_title(title)
    if len(vals) <= 32:
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(labels_for(n), rotation=90 if n > 3 else 0, fontsize=8)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    return ax

def plot_probs(state, title="probabilities", mark=None, ax=None):
    """วาดความน่าจะเป็น |amp|^2"""
    state = np.asarray(state, dtype=complex)
    n = nqubits(state)
    p = np.abs(state) ** 2
    if ax is None:
        _, ax = plt.subplots(figsize=(min(10, 1.0 + 0.45 * len(p)), 2.7))
    colors = [POS] * len(p)
    if mark is not None:
        mark = set(mark if hasattr(mark, "__iter__") else [mark])
        colors = [MARK if i in mark else c for i, c in enumerate(colors)]
    ax.bar(range(len(p)), p, color=colors, width=0.68)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("probability")
    ax.set_title(title)
    if len(p) <= 32:
        ax.set_xticks(range(len(p)))
        ax.set_xticklabels(labels_for(n), rotation=90 if n > 3 else 0, fontsize=8)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    return ax

def plot_counts(counts, title="measured counts", ax=None):
    """วาดผลนับจากการวัดจริง"""
    keys = sorted(counts)
    vals = [counts[k] for k in keys]
    if ax is None:
        _, ax = plt.subplots(figsize=(min(10, 1.2 + 0.7 * len(keys)), 2.7))
    ax.bar(range(len(keys)), vals, color=POS, width=0.6)
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels(keys, rotation=90 if len(keys) > 8 else 0, fontsize=9)
    ax.set_ylabel("shots")
    ax.set_title(title)
    total = sum(vals)
    for i, v in enumerate(vals):
        ax.text(i, v, " {:.1%}".format(v / total), ha="center", va="bottom",
                fontsize=8, color=INK, rotation=90 if len(keys) > 8 else 0)
    ax.set_ylim(0, max(vals) * 1.22)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    return ax

print("พร้อมแล้ว · numpy", np.__version__)

---
# ตอนที่ 1 — เวกเตอร์และจำนวนเชิงซ้อนใน numpy

ก่อนจะพูดถึงคิวบิต ต้องคล่องสองอย่างนี้ก่อน

**เวกเตอร์** คือรายการตัวเลข ใน numpy คือ `np.array([...])`

**จำนวนเชิงซ้อน** คือตัวเลขที่มีสองส่วน `a + bj` (Python ใช้ `j` ไม่ใช่ `i`)
วิธีที่ดีที่สุดในการคิดเรื่องจำนวนเชิงซ้อนคือมองเป็น **ลูกศรในระนาบ** ที่มี *ขนาด* กับ *มุม*
และการคูณจำนวนเชิงซ้อนคือ **ขนาดคูณกัน มุมบวกกัน**

In [ ]:
# จำนวนเชิงซ้อนใน Python ใช้ j ไม่ใช่ i
z = 3 + 4j
print("z        =", z)
print("ขนาด |z| =", abs(z))                 # = 5 (ทฤษฎีบทพีทาโกรัส)
print("มุม arg z =", np.angle(z, deg=True), "องศา")
print("คอนจูเกต  =", np.conj(z))             # พลิกเครื่องหมายส่วนจินตภาพ

print()
# คูณด้วย e^(i*phi) = หมุนอย่างเดียว ไม่เปลี่ยนขนาด
phi = np.pi / 2                              # 90 องศา
w = np.exp(1j * phi)
print("w = e^(i*pi/2) =", np.round(w, 10), "  <- ก็คือ i นั่นเอง")
print("z*w            =", np.round(z * w, 10))
print("ขนาดเท่าเดิม    =", abs(z * w))
print("มุมเพิ่มมา 90    =", np.angle(z * w, deg=True), "องศา")

In [ ]:
# เวกเตอร์เชิงซ้อน
a = np.array([1 + 1j, 2 - 1j])
b = np.array([0 + 1j, 3 + 0j])

print("a =", a)
print("b =", b)
print()

# ผลคูณภายใน <a|b> = sum( conj(a) * b )
# สังเกตว่าตัวซ้ายต้องคอนจูเกตก่อน -- นี่คือจุดที่ต่างจาก dot product ของจำนวนจริง
inner = np.vdot(a, b)          # np.vdot คอนจูเกตตัวแรกให้อัตโนมัติ
print("<a|b> =", inner)
print("ทำมือ  =", np.sum(np.conj(a) * b))

# ความยาว (นอร์ม) = sqrt(<a|a>)
print()
print("|a| =", np.linalg.norm(a), " ตรวจ:", np.sqrt(np.vdot(a, a).real))

### ทำไมต้องคอนจูเกต

ถ้าไม่คอนจูเกต ความยาวกำลังสองอาจติดลบหรือเป็นจำนวนเชิงซ้อนได้ ซึ่งใช้เป็นความน่าจะเป็นไม่ได้
ลองดู: ถ้า `a = [i]` แล้ว `a·a = i*i = -1` แต่ `<a|a> = conj(i)*i = 1` ✓

**กฎทอง:** ในควอนตัม ความยาวของเวกเตอร์สถานะต้องเป็น 1 เสมอ เพราะความยาวกำลังสองคือผลรวมความน่าจะเป็น

In [ ]:
# ━━━ แบบฝึกหัด 1.1 ━━━
# 1) สร้างเวกเตอร์ v = [1+2j, 2-1j, 0+1j]
# 2) ทำให้มันยาว 1 (normalize) แล้วตั้งชื่อว่า v_hat
# 3) ตรวจว่า <v_hat|v_hat> = 1 จริง

v = None      # <- เขียนตรงนี้
v_hat = None  # <- เขียนตรงนี้

In [ ]:
#@title ✅ เฉลย 1.1 { display-mode: "form" }
v = np.array([1 + 2j, 2 - 1j, 0 + 1j])
v_hat = v / np.linalg.norm(v)
print("v     =", v)
print("v_hat =", v_hat)
print("<v_hat|v_hat> =", np.vdot(v_hat, v_hat).real, " <- ต้องได้ 1")

---
# ตอนที่ 2 — คิวบิตคืออะไร

> สไลด์หน้า 4, 8

**บิตคลาสสิก** เลือกได้สองค่า: 0 หรือ 1

**คิวบิต** คือ **เวกเตอร์หนึ่งหน่วยใน $\mathbb{C}^2$**

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad |\alpha|^2 + |\beta|^2 = 1$$

โดย $\alpha, \beta$ เป็น **จำนวนเชิงซ้อน** เรียกว่า **แอมพลิจูด**

สัญกรณ์ `|0>` อ่านว่า "เคตศูนย์" (ket) มันคือเวกเตอร์ `[1, 0]` เฉย ๆ ไม่มีอะไรลึกลับ

**ข้อควรระวังที่สำคัญที่สุดของทั้ง notebook:** แอมพลิจูด **ไม่ใช่** ความน่าจะเป็น
ความน่าจะเป็นคือ $|\alpha|^2$ ต่างหาก และแอมพลิจูด **ติดลบหรือเป็นจำนวนเชิงซ้อนได้** ซึ่งความน่าจะเป็นทำไม่ได้
ความต่างข้อนี้คือที่มาของพลังทั้งหมดของควอนตัม (เดี๋ยวเจอในตอนที่ 14–15)

In [ ]:
# สองสถานะฐาน -- แค่เวกเตอร์ธรรมดา
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

print("|0> =", ket0)
print("|1> =", ket1)
print("<0|1> =", np.vdot(ket0, ket1).real, " <- ตั้งฉากกัน จึงแยกแยะได้ 100%")
print()

# สถานะซ้อนทับที่ใช้บ่อย
plus   = (ket0 + ket1) / np.sqrt(2)      # |+>
minus  = (ket0 - ket1) / np.sqrt(2)      # |->
plus_i = (ket0 + 1j * ket1) / np.sqrt(2) # |i>

for name, s in [("|+>", plus), ("|->", minus), ("|i>", plus_i)]:
    print("{:5s} = {}   norm = {:.3f}".format(name, np.round(s, 3), np.linalg.norm(s)))

In [ ]:
def qubit(theta, phi=0.0):
    """
    สร้างคิวบิตจากสองมุมบนทรงกลมบลอค (สไลด์หน้า 8)
        |psi> = cos(theta/2)|0> + e^(i*phi) sin(theta/2)|1>
    theta : 0..pi   มุมจากขั้วเหนือ (คุมความน่าจะเป็น)
    phi   : 0..2pi  เฟสสัมพัทธ์     (ไม่คุมความน่าจะเป็นเลย!)
    """
    return np.array([np.cos(theta / 2),
                     np.exp(1j * phi) * np.sin(theta / 2)], dtype=complex)

print("theta=0      ->", np.round(qubit(0), 3),            "  = |0>")
print("theta=pi     ->", np.round(qubit(np.pi), 3),        "  = |1>")
print("theta=pi/2   ->", np.round(qubit(np.pi/2), 3),      "  = |+>")
print("theta=pi/2, phi=pi ->", np.round(qubit(np.pi/2, np.pi), 3), "  = |->")

print()
print("--- จุดสำคัญ: เฟสเปลี่ยน แต่ความน่าจะเป็นไม่ขยับ ---")
for phi in [0, np.pi/2, np.pi, 3*np.pi/2]:
    s = qubit(np.pi/2, phi)
    print("phi={:5.2f}  state={}   P(0)={:.3f}  P(1)={:.3f}".format(
        phi, np.round(s, 3), abs(s[0])**2, abs(s[1])**2))

In [ ]:
# วาดให้เห็นว่า |+> กับ |-> ต่างกันจริง ทั้งที่ความน่าจะเป็นเหมือนกันเป๊ะ
fig, axes = plt.subplots(2, 2, figsize=(8, 5))
plot_amplitudes(plus,  "|+>  amplitudes", ax=axes[0][0])
plot_amplitudes(minus, "|->  amplitudes", ax=axes[0][1])
plot_probs(plus,  "|+>  probabilities", ax=axes[1][0])
plot_probs(minus, "|->  probabilities", ax=axes[1][1])
plt.tight_layout(); plt.show()

print("แถวบน (แอมพลิจูด) ต่างกันชัดเจน -- แท่งขวาชี้คนละทาง")
print("แถวล่าง (ความน่าจะเป็น) เหมือนกันเป๊ะ")
print("=> ข้อมูลที่แยกสองสถานะนี้ออกจากกัน ซ่อนอยู่ในเครื่องหมาย ซึ่งการวัดตรง ๆ มองไม่เห็น")

In [ ]:
# ━━━ แบบฝึกหัด 2.1 ━━━
# สร้างสถานะที่มีโอกาสวัดได้ 0 เท่ากับ 90% และวัดได้ 1 เท่ากับ 10%
# คำใบ้: P(0) = cos^2(theta/2) -> หา theta จาก arccos
state_90 = None   # <- เขียนตรงนี้

In [ ]:
#@title ✅ เฉลย 2.1 { display-mode: "form" }
theta = 2 * np.arccos(np.sqrt(0.9))
state_90 = qubit(theta)
print("theta =", np.degrees(theta), "องศา")
print("state =", np.round(state_90, 4))
print("P(0) = {:.3f}   P(1) = {:.3f}".format(abs(state_90[0])**2, abs(state_90[1])**2))

---
# ตอนที่ 3 — การวัดและกฎของบอร์น

> สไลด์หน้า 6–7

ทุกอย่างก่อนหน้านี้เป็นพีชคณิตเชิงเส้นล้วน ๆ **จุดเดียวที่ควอนตัมต่างออกไปจริง ๆ คือตรงนี้**

**กฎของบอร์น:** ถ้าวัดสถานะ $|\psi\rangle$ ในฐานตั้งฉาก $\{|b_0\rangle, |b_1\rangle\}$
โอกาสได้ผลเป็น $b_k$ คือ

$$P(b_k) = |\langle b_k | \psi \rangle|^2$$

และ **หลังวัดเสร็จ สถานะจะยุบไปเป็น $|b_k\rangle$ ทันที** ข้อมูลเดิมหายไปตลอดกาล

นี่คือเหตุผลที่เราอ่านค่า $\alpha, \beta$ ออกมาตรง ๆ ไม่ได้ ต้องเตรียมสถานะใหม่แล้วยิงซ้ำหลาย ๆ ครั้ง

In [ ]:
def probabilities(state):
    """ความน่าจะเป็นของทุกผลลัพธ์ในฐานคำนวณ {|0>, |1>, ...}"""
    p = np.abs(np.asarray(state, dtype=complex)) ** 2
    return p / p.sum()

def measure(state, shots=1000, seed=None):
    """จำลองการยิงเครื่องวัด shots ครั้ง (เตรียมสถานะใหม่ทุกครั้ง) คืน dict ของผลนับ"""
    rng = np.random.default_rng(seed)
    p = probabilities(state)
    n = nqubits(state)
    draws = rng.choice(len(p), size=shots, p=p)
    counts = {}
    for d in draws:
        key = format(d, "0{}b".format(n))
        counts[key] = counts.get(key, 0) + 1
    return dict(sorted(counts.items()))

psi = qubit(np.pi / 3)          # เอียงไปทาง |0> มากกว่า
print("สถานะ :", np.round(psi, 3))
print("ทฤษฎี :", np.round(probabilities(psi), 4))

c = measure(psi, shots=2000, seed=1)
print("วัดจริง:", c)
plot_counts(c, "2000 shots"); plt.show()

In [ ]:
# วัดในฐานอื่น -- นี่คือ "ฐานตั้งฉาก" จากพีชคณิตเชิงเส้นมาเต็ม ๆ
BASES = {
    "Z": (ket0, ket1,  "|0>", "|1>"),      # ฐานคำนวณ
    "X": (plus, minus, "|+>", "|->"),      # ไอเกนสเตตของ sigma_x
    "Y": ((ket0 + 1j*ket1)/np.sqrt(2), (ket0 - 1j*ket1)/np.sqrt(2), "|i>", "|-i>"),
}

def measure_in_basis(state, basis="Z"):
    b0, b1, n0, n1 = BASES[basis]
    p0 = abs(np.vdot(b0, state)) ** 2
    p1 = abs(np.vdot(b1, state)) ** 2
    return {n0: p0, n1: p1}

print("ทดลองสำคัญ: สถานะเดียวกัน วัดคนละฐาน ได้คนละเรื่อง")
print()
for name, s in [("|0>", ket0), ("|+>", plus), ("|->", minus)]:
    print("สถานะ", name)
    for b in ["Z", "X", "Y"]:
        r = measure_in_basis(s, b)
        print("   ฐาน {} :  {}".format(b, {k: round(v, 3) for k, v in r.items()}))
    print()

### อ่านผลข้างบนให้ดี

- `|+>` วัดในฐาน **Z** → สุ่มมั่ว 50/50
- `|+>` วัดในฐาน **X** → ได้ `|+>` ทุกครั้ง **100%** ไม่พลาดเลย

สถานะเดียวกันแท้ ๆ แต่ "สุ่ม" หรือ "แน่นอน" ขึ้นกับว่าเรา **ถามด้วยฐานไหน**

ความสุ่มในควอนตัมจึงไม่ใช่สมบัติของสถานะเพียงอย่างเดียว แต่เป็นความสัมพันธ์ระหว่าง *สถานะ* กับ *คำถามที่เราถาม*

In [ ]:
# ━━━ แบบฝึกหัด 3.1 ━━━
# สถานะ |psi> = (1/sqrt(3))|0> + sqrt(2/3)|1>   (ตัวอย่างในสไลด์หน้า 7)
# 1) P(0) กับ P(1) เป็นเท่าไร
# 2) ยิง 3000 ครั้งแล้ววาดกราฟ เทียบกับค่าทฤษฎี

In [ ]:
#@title ✅ เฉลย 3.1 { display-mode: "form" }
psi_slide = np.array([1/np.sqrt(3), np.sqrt(2/3)], dtype=complex)
print("สถานะ:", np.round(psi_slide, 4), " norm =", np.linalg.norm(psi_slide))
print("ทฤษฎี: P(0) = 1/3 = {:.4f}   P(1) = 2/3 = {:.4f}".format(1/3, 2/3))
c = measure(psi_slide, 3000, seed=7)
print("วัดจริง:", {k: "{} ({:.1%})".format(v, v/3000) for k, v in c.items()})
plot_counts(c, "psi = (1/sqrt3)|0> + sqrt(2/3)|1>  ·  3000 shots"); plt.show()

---
# ตอนที่ 4 — ทรงกลมบลอค

> สไลด์หน้า 8–9

$\mathbb{C}^2$ มีสี่มิติจริง วาดไม่ได้ แต่พอตัดเงื่อนไขสองข้อออก
(ความยาวต้องเป็น 1 และ *เฟสรวม* ที่วัดไม่เห็น) เหลืออิสระแค่สองมุม — วาดเป็นผิวทรงกลมได้พอดี

$$\vec{r} = \big(\sin\theta\cos\varphi,\ \sin\theta\sin\varphi,\ \cos\theta\big)$$

**กับดักที่สไลด์หน้า 9 เตือนไว้:** บนทรงกลมบลอค มุมถูก *เบิ้ล* ไว้
`|0>` กับ `|1>` ตั้งฉากกันใน Hilbert space (90°) แต่บนทรงกลมมันอยู่คนละขั้วเลย (180°)

In [ ]:
def bloch_vector(state):
    """แปลงสถานะคิวบิตเป็นพิกัด (x, y, z) บนทรงกลมบลอค
       สูตรนี้ไม่สนใจเฟสรวม จึงให้ผลเดียวกันสำหรับ |psi> และ e^(i*g)|psi>"""
    a, b = np.asarray(state, dtype=complex)
    ab = np.conj(a) * b
    return np.array([2 * ab.real, 2 * ab.imag, abs(a) ** 2 - abs(b) ** 2])

def plot_bloch(states, names=None, title=""):
    """วาดทรงกลมบลอคพร้อมลูกศรสถานะ"""
    if isinstance(states, np.ndarray) and states.ndim == 1:
        states = [states]
    names = list(names) if names is not None else []
    names += [""] * (len(states) - len(names))
    fig = plt.figure(figsize=(4.4, 4.4))
    ax = fig.add_subplot(111, projection="3d")
    u, v = np.mgrid[0:2*np.pi:33j, 0:np.pi:17j]
    ax.plot_wireframe(np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v),
                      color=GRID, linewidth=0.4)
    for p, lab in [((0,0,1.25), "|0>"), ((0,0,-1.25), "|1>"),
                   ((1.3,0,0), "|+>"), ((-1.3,0,0), "|->"),
                   ((0,1.3,0), "|i>"), ((0,-1.3,0), "|-i>")]:
        ax.text(*p, lab, color=MUTED, fontsize=8, ha="center")
    for ax_v in [(1,0,0), (0,1,0), (0,0,1)]:
        ax.plot(*[[-a, a] for a in ax_v], color=MUTED, lw=0.7, ls=":")
    cols = [MARK, POS, NEG, "#1baf7a"]
    for k, s in enumerate(states):
        r = bloch_vector(s)
        ax.quiver(0, 0, 0, *r, color=cols[k % len(cols)], lw=2.6,
                  arrow_length_ratio=0.13)
        if names[k]:
            ax.text(*(r * 1.12), names[k], color=cols[k % len(cols)],
                    fontsize=9, weight="bold")
    ax.set_box_aspect([1, 1, 1]); ax.set_axis_off()
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
    ax.set_title(title, fontsize=10)
    plt.tight_layout(); plt.show()

for name, s in [("|0>", ket0), ("|1>", ket1), ("|+>", plus), ("|i>", plus_i)]:
    print("{:5s} -> bloch {}".format(name, np.round(bloch_vector(s), 3)))

plot_bloch([ket0, plus, plus_i], ["|0>", "|+>", "|i>"], "three basis states")

In [ ]:
# เฟสรวมไม่มีผลต่ออะไรที่สังเกตได้เลย -- พิสูจน์ด้วยโค้ด
s1 = plus
s2 = np.exp(1j * 1.234) * plus      # คูณเฟสรวมมั่ว ๆ

print("สถานะ 1 :", np.round(s1, 3))
print("สถานะ 2 :", np.round(s2, 3), " <- หน้าตาต่างกันมาก")
print()
print("เวกเตอร์บลอคเหมือนกันไหม :", np.allclose(bloch_vector(s1), bloch_vector(s2)))
print("ความน่าจะเป็นเหมือนกันไหม:", np.allclose(probabilities(s1), probabilities(s2)))
print()
print("=> เฟสรวม (global phase) วัดไม่ได้ จึงถือว่าเป็นสถานะเดียวกัน")
print("   แต่เฟส *สัมพัทธ์* ระหว่าง |0> กับ |1> วัดได้ (ด้วยการเปลี่ยนฐาน) และสำคัญมาก")

In [ ]:
# ━━━ แบบฝึกหัด 4.1 ━━━
# หาสถานะที่เวกเตอร์บลอคชี้ไปทาง (0, -1, 0) แล้ววาดบนทรงกลม

In [ ]:
#@title ✅ เฉลย 4.1 { display-mode: "form" }
# (0,-1,0) คือแกน -y ซึ่งตรงกับ theta = pi/2, phi = -pi/2  (หรือ 3pi/2)
minus_i = qubit(np.pi/2, -np.pi/2)
print("สถานะ:", np.round(minus_i, 3), "  = (|0> - i|1>)/sqrt(2)")
print("bloch :", np.round(bloch_vector(minus_i), 3))
plot_bloch([minus_i], ["|-i>"], "")

---
# ตอนที่ 5 — ประตูควอนตัม 1 คิวบิต

> สไลด์หน้า 10–11

ประตูควอนตัมคือ **เมทริกซ์ยูนิแทรี** คือเมทริกซ์เชิงซ้อนที่

$$U^\dagger U = I$$

โดย $U^\dagger$ คือ "ทรานสโพสแล้วคอนจูเกต" (dagger) เงื่อนไขนี้ให้ผลสามอย่างพร้อมกัน:

1. **ความยาวไม่เปลี่ยน** → ความน่าจะเป็นรวมยังเป็น 1 เสมอ
2. **มุมระหว่างสถานะไม่เปลี่ยน** → สถานะที่แยกแยะได้ ก็ยังแยกแยะได้
3. **ย้อนกลับได้เสมอ** เพราะ $U^{-1} = U^\dagger$ → วงจรควอนตัมทิ้งข้อมูลไม่ได้

ข้อ 3 คือเหตุผลที่ไม่มีประตู AND หรือ OR แบบคลาสสิกในวงจรควอนตัม

In [ ]:
# ประตูมาตรฐานตามสไลด์หน้า 10
I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)              # bit flip  (NOT)
Y  = np.array([[0, -1j], [1j, 0]], dtype=complex)           # bit & phase flip
Z  = np.array([[1, 0], [0, -1]], dtype=complex)             # phase flip
H  = np.array([[1, 1], [1, -1]], dtype=complex)/np.sqrt(2)  # สร้างการซ้อนทับ
S  = np.array([[1, 0], [0, 1j]], dtype=complex)             # เฟส 90
T  = np.array([[1, 0], [0, np.exp(1j*np.pi/4)]], dtype=complex)  # เฟส 45 (pi/8 gate)

GATES = {"I": I2, "X": X, "Y": Y, "Z": Z, "H": H, "S": S, "T": T}

def dagger(U):
    return U.conj().T

def is_unitary(U, tol=1e-10):
    return np.allclose(dagger(U) @ U, np.eye(U.shape[0]), atol=tol)

print("ตรวจความเป็นยูนิแทรีของทุกประตู")
for name, U in GATES.items():
    print("  {}:  U+U = I ? {}".format(name, is_unitary(U)))

In [ ]:
# ประตูทำอะไรกับสถานะบ้าง
print("ผลของแต่ละประตูเมื่อกระทำกับ |0>")
for name in ["I", "X", "Y", "Z", "H", "S", "T"]:
    out = GATES[name] @ ket0
    print("  {} |0> = {}   bloch {}".format(name, np.round(out, 3),
                                            np.round(bloch_vector(out), 3)))

print()
print("H คือตัวสลับระหว่างฐาน Z กับฐาน X")
print("  H|0> =", np.round(H @ ket0, 3), " = |+>")
print("  H|1> =", np.round(H @ ket1, 3), " = |->")
print("  H|+> =", np.round(H @ plus, 3), " = |0>  (ทำสองครั้งได้ตัวเดิมคืน)")

plot_bloch([ket0, H @ ket0, X @ ket0], ["|0>", "H|0>", "X|0>"], "gate effects on |0>")

In [ ]:
# ประตูหมุนแบบปรับมุมได้ -- ฮาร์ดแวร์จริงควบคุมด้วยประตูชนิดนี้
def RX(t): return np.array([[np.cos(t/2), -1j*np.sin(t/2)],
                            [-1j*np.sin(t/2), np.cos(t/2)]], dtype=complex)
def RY(t): return np.array([[np.cos(t/2), -np.sin(t/2)],
                            [np.sin(t/2),  np.cos(t/2)]], dtype=complex)
def RZ(t): return np.array([[np.exp(-1j*t/2), 0],
                            [0, np.exp(1j*t/2)]], dtype=complex)

print("Rx(pi) เท่ากับ X ไหม (ขึ้นกับเฟสรวม)")
print(np.round(RX(np.pi), 3))
print("X =")
print(np.round(X, 3))
print("ต่างกันแค่ตัวคูณ -i :", np.allclose(RX(np.pi), -1j * X))

print()
print("ค่อย ๆ หมุน |0> รอบแกน y ทีละนิด")
for t in [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]:
    s = RY(t) @ ket0
    print("  Ry({:4.2f}) |0> : P(0)={:.3f}  bloch z={:+.3f}".format(
        t, abs(s[0])**2, bloch_vector(s)[2]))

In [ ]:
# ━━━ แบบฝึกหัด 5.1 ━━━
# พิสูจน์ด้วยโค้ดว่า  H Z H = X
# คำใบ้: ใน numpy การคูณเมทริกซ์ใช้ @  และระวังลำดับ!

In [ ]:
#@title ✅ เฉลย 5.1 { display-mode: "form" }
print("H @ Z @ H =")
print(np.round(H @ Z @ H, 10))
print("X =")
print(np.round(X, 10))
print()
print("เท่ากันไหม:", np.allclose(H @ Z @ H, X))
print()
print("ความหมาย: Z พลิกเฟสในฐาน Z แต่เมื่อมองผ่านฐาน X (ซึ่ง H พาไป)")
print("          มันกลายเป็นการพลิกบิตธรรมดา -- ฐานต่างกัน ความหมายต่างกัน")

---
# ตอนที่ 6 — วงจร = การคูณเมทริกซ์

วงจรควอนตัมคือลำดับประตู อ่านจากซ้ายไปขวา
แต่เมทริกซ์รวมเขียน **กลับทาง** เพราะตัวที่ทำก่อนต้องอยู่ติดกับเวกเตอร์

$$\text{วงจร } G_1 \to G_2 \to G_3 \quad\Longleftrightarrow\quad U = G_3 G_2 G_1$$

ไม่ว่าวงจรจะยาวแค่ไหน มัน **ยุบเหลือเมทริกซ์ 2×2 ตัวเดียวได้เสมอ**

In [ ]:
def circuit(gate_names):
    """ยุบลำดับประตู (ซ้ายไปขวา) ให้เหลือเมทริกซ์เดียว"""
    U = np.eye(2, dtype=complex)
    for g in gate_names:
        U = GATES[g] @ U          # ตัวใหม่คูณจากทางซ้าย
    return U

def run(gate_names, state=None, show=True):
    state = ket0.copy() if state is None else state
    U = circuit(gate_names)
    out = U @ state
    if show:
        print("วงจร :", " -> ".join(gate_names) if gate_names else "(ว่าง)")
        print("U    =\n", np.round(U, 3))
        print("ออก  =", np.round(out, 3), "   P =", np.round(probabilities(out), 3))
        print()
    return out

run(["H"])
run(["H", "H"])            # ได้ |0> คืน
run(["H", "Z", "H"])       # = X -> ได้ |1>
run(["T"] * 8)             # T แปดครั้ง = I

In [ ]:
# เส้นทางบนทรงกลมบลอคระหว่างเดินผ่านวงจร
seq = ["H", "T", "T", "H", "S"]
state = ket0.copy()
path = [state]
for g in seq:
    state = GATES[g] @ state
    path.append(state)

print("เดินผ่านวงจร:", " -> ".join(seq))
for g, s in zip(["เริ่ม"] + seq, path):
    print("  {:5s}  P(0)={:.3f}  bloch={}".format(g, abs(s[0])**2, np.round(bloch_vector(s), 2)))

plot_bloch(path[::2], ["start", seq[1], seq[3]], "every other step of the circuit")

In [ ]:
# ━━━ แบบฝึกหัด 6.1 ━━━
# หาวงจรสั้น ๆ ที่เปลี่ยน |0> ให้กลายเป็น |i> = (|0> + i|1>)/sqrt(2)
# คำใบ้: ต้องสร้างการซ้อนทับก่อน แล้วค่อยใส่เฟส 90 องศา

In [ ]:
#@title ✅ เฉลย 6.1 { display-mode: "form" }
out = run(["H", "S"])
print("เป้าหมาย |i> =", np.round(plus_i, 3))
print("ตรงกันไหม   :", np.allclose(out, plus_i))
print()
print("เหตุผล: H สร้างซ้อนทับ -> (|0>+|1>)/sqrt2 แล้ว S ใส่เฟส i ให้เฉพาะ |1>")

---
# ตอนที่ 7 — เมทริกซ์ความหนาแน่น

> สไลด์หน้า 4–5

$|\psi\rangle$ ใช้ได้เฉพาะตอนที่เรารู้สถานะแน่นอน
แต่ถ้าเครื่องเตรียมสถานะมั่ว ๆ — 50% เป็น `|0>` 50% เป็น `|1>` — เขียนเป็นเวกเตอร์ตัวเดียวไม่ได้

ต้องใช้ **เมทริกซ์ความหนาแน่น** ซึ่งสร้างจาก **ket-bra** (ผลคูณภายนอก)

$$\rho = |\psi\rangle\langle\psi|$$

ความต่างสำคัญ:
- `<b|a>` = แถว × คอลัมน์ = **ตัวเลขหนึ่งตัว**
- `|a><b|` = คอลัมน์ × แถว = **เมทริกซ์ 2×2**

ตัววัดว่าบริสุทธิ์หรือผสมคือ $\mathrm{Tr}(\rho^2)$ — เท่ากับ 1 คือบริสุทธิ์ น้อยกว่า 1 คือผสม

In [ ]:
def ketbra(a, b=None):
    """|a><b|  ถ้าไม่ใส่ b จะใช้ a เอง"""
    b = a if b is None else b
    return np.outer(a, np.conj(b))

def purity(rho):
    return np.trace(rho @ rho).real

print("--- bra-ket ได้ตัวเลข ---")
print("<0|1> =", np.vdot(ket0, ket1))
print()
print("--- ket-bra ได้เมทริกซ์ ---")
print("|0><0| =\n", np.round(ketbra(ket0), 3))
print("|0><1| =\n", np.round(ketbra(ket0, ket1), 3))

In [ ]:
# ตัวอย่างสามตัวจากสไลด์หน้า 5
rho_pure0 = ketbra(ket0)                               # บริสุทธิ์
rho_mixed = 0.5 * ketbra(ket0) + 0.5 * ketbra(ket1)    # ผสมสูงสุด = I/2
rho_minus = ketbra(minus)                              # บริสุทธิ์ (แต่หน้าตาคล้ายผสม!)

for name, r in [("|0><0|", rho_pure0), ("(|0><0| + |1><1|)/2", rho_mixed), ("|-><-|", rho_minus)]:
    print(name)
    print(np.round(r, 3))
    print("   Tr(rho)  = {:.3f}   <- ต้องเป็น 1 เสมอ".format(np.trace(r).real))
    print("   Tr(rho^2)= {:.3f}   -> {}".format(purity(r), "บริสุทธิ์" if purity(r) > 0.999 else "ผสม"))
    print()

In [ ]:
# เส้นทางจาก "บริสุทธิ์" ไป "ผสม" -- ภาพของ decoherence
def rho_from_bloch(r):
    """rho = (I + r.sigma)/2  : เวกเตอร์บลอคสั้นลง = ผสมมากขึ้น"""
    x, y, z = r
    return 0.5 * np.array([[1 + z, x - 1j*y], [x + 1j*y, 1 - z]], dtype=complex)

ps = np.linspace(1, 0, 6)
print("ยิ่ง p เล็ก = ยิ่งผสม  (p คือความยาวเวกเตอร์บลอค)")
print()
print("  p      rho01 (coherence)   Tr(rho^2)   P(0)")
for p in ps:
    r = rho_from_bloch([p, 0, 0])          # ทิศ |+>
    print("  {:.1f}      {:.3f}             {:.3f}     {:.3f}".format(
        p, abs(r[0, 1]), purity(r), r[0, 0].real))

print()
print("สังเกต: P(0) ค้างที่ 0.5 ตลอด แต่ coherence (ตัวนอกเส้นทแยง) หดหายไป")
print("        นั่นคือ decoherence -- ความเป็นควอนตัมหายไป เหลือแค่ความสุ่มคลาสสิก")

fig, ax = plt.subplots(figsize=(5, 2.8))
pp = np.linspace(0, 1, 60)
ax.plot(pp, [abs(rho_from_bloch([p,0,0])[0,1]) for p in pp], color=POS, lw=2.2, label="coherence |rho01|")
ax.plot(pp, [purity(rho_from_bloch([p,0,0])) for p in pp], color=NEG, lw=2.2, label="purity Tr(rho^2)")
ax.set_xlabel("Bloch vector length p"); ax.legend(frameon=False, fontsize=9)
ax.yaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.set_title("pure (p=1)  ->  maximally mixed (p=0)")
plt.tight_layout(); plt.show()

In [ ]:
# ━━━ แบบฝึกหัด 7.1 ━━━
# สไลด์หน้า 5 บอกว่า rho = (1/2)[[1,-1],[-1,1]] เป็นสถานะ "บริสุทธิ์"
# จงยืนยันด้วยโค้ด และหาว่ามันคือ |psi><psi| ของสถานะ psi ตัวไหน

In [ ]:
#@title ✅ เฉลย 7.1 { display-mode: "form" }
rho = 0.5 * np.array([[1, -1], [-1, 1]], dtype=complex)
print("rho =\n", np.round(rho, 3))
print("Tr(rho)   =", np.trace(rho).real)
print("Tr(rho^2) =", purity(rho), " -> บริสุทธิ์ เพราะเท่ากับ 1")
print()
# หาสถานะด้วยไอเกนเวกเตอร์ที่มีไอเกนแวลู 1
vals, vecs = np.linalg.eigh(rho)
psi = vecs[:, np.argmax(vals)]
print("ไอเกนแวลูของ rho :", np.round(vals, 6), " <- บริสุทธิ์คือมี 1 ตัวเดียวที่เป็น 1 ที่เหลือเป็น 0")
print("ไอเกนเวกเตอร์    :", np.round(psi, 4))
print("เทียบกับ |->     :", np.round(minus, 4), " (ต่างกันได้แค่เครื่องหมายรวม)")
print("rho == |-><-| ?  :", np.allclose(rho, ketbra(minus)))

---
# ตอนที่ 8 — ผลคูณเทนเซอร์ · ระบบหลายคิวบิต

> สไลด์หน้า 12

รวมสองระบบใช้ **ผลคูณเทนเซอร์** $\otimes$ ไม่ใช่การเอามาต่อกัน

$$|a\rangle \otimes |b\rangle = \begin{pmatrix}a_1\\a_2\end{pmatrix} \otimes \begin{pmatrix}b_1\\b_2\end{pmatrix} = \begin{pmatrix}a_1b_1\\a_1b_2\\a_2b_1\\a_2b_2\end{pmatrix}$$

คิวบิตละ 2 มิติ รวมกันได้ **4 มิติ** ไม่ใช่ 4 แบบบวก แต่เป็นแบบ **คูณ**

นี่คือที่มาของพลังทั้งหมด: $n$ คิวบิตต้องใช้ตัวเลข $2^n$ ตัวบรรยาย
ที่ 300 คิวบิต จำนวนตัวเลขก็เกินจำนวนอะตอมในเอกภพแล้ว

In [ ]:
# numpy มี np.kron ให้ใช้เลย
a = np.array([0.6, 0.8], dtype=complex)
b = np.array([1, 0], dtype=complex)

print("a      =", a)
print("b      =", b)
print("a (x) b=", np.round(np.kron(a, b), 3))
print()

# ตัวอย่างจากสไลด์: |1>_A (x) |0>_B
print("|1> (x) |0> =", np.real(np.kron(ket1, ket0)).astype(int), " = |10>")
print("ลำดับช่องคือ:", labels_for(2))

In [ ]:
def tensor(*states):
    """ผลคูณเทนเซอร์ของหลายสถานะ"""
    out = states[0]
    for s in states[1:]:
        out = np.kron(out, s)
    return out

def op_on(U, q, n):
    """ขยายประตู 1 คิวบิตเป็นเมทริกซ์ 2^n x 2^n ที่กระทำกับคิวบิตที่ q
       วิธีนี้ 'เห็นภาพ' ชัด แต่ช้าเมื่อ n ใหญ่"""
    ops = [np.eye(2, dtype=complex)] * n
    ops[q] = U
    out = ops[0]
    for o in ops[1:]:
        out = np.kron(out, o)
    return out

print("H บนคิวบิตที่ 0 ของระบบ 2 คิวบิต  =  H (x) I")
print(np.round(op_on(H, 0, 2), 3))
print()
print("ลองใช้: |00> -> ?")
s = op_on(H, 0, 2) @ tensor(ket0, ket0)
print(np.round(s, 3), " = (|00> + |10>)/sqrt(2)")

In [ ]:
# เวอร์ชันเร็ว: ไม่สร้างเมทริกซ์ใหญ่ ใช้เทคนิคปรับรูปร่างอาเรย์แทน
# (จำเป็นเมื่อ n มาก เช่นตอนที่ 15 ที่ใช้ 10 คิวบิต)
def apply_1q(state, U, q, n):
    psi = np.asarray(state, dtype=complex).reshape([2] * n)
    psi = np.moveaxis(psi, q, 0)
    shp = psi.shape
    psi = (U @ psi.reshape(2, -1)).reshape(shp)
    return np.moveaxis(psi, 0, q).reshape(-1)

def zero_state(n):
    s = np.zeros(2 ** n, dtype=complex); s[0] = 1.0
    return s

def basis_state(n, k):
    s = np.zeros(2 ** n, dtype=complex); s[k] = 1.0
    return s

# ตรวจว่าสองวิธีให้ผลเหมือนกัน
s_slow = op_on(H, 1, 3) @ zero_state(3)
s_fast = apply_1q(zero_state(3), H, 1, 3)
print("สองวิธีตรงกันไหม:", np.allclose(s_slow, s_fast))
print()

# สร้างการซ้อนทับทุกสถานะพร้อมกัน -- H ทุกคิวบิต
n = 3
s = zero_state(n)
for q in range(n):
    s = apply_1q(s, H, q, n)
print("H ทุกคิวบิตของ |000> :")
print(np.round(s.real, 3))
print("=> ทุกสถานะมีแอมพลิจูดเท่ากัน = 1/sqrt(8) =", round(1/np.sqrt(8), 3))
plot_amplitudes(s, "uniform superposition over 8 states"); plt.show()

In [ ]:
# ━━━ แบบฝึกหัด 8.1 ━━━
# สร้างสถานะ 3 คิวบิตที่คิวบิตแรกเป็น |1> ส่วนอีกสองตัวอยู่ในการซ้อนทับ |+>
# แล้วดูว่าความน่าจะเป็นกระจายยังไง

In [ ]:
#@title ✅ เฉลย 8.1 { display-mode: "form" }
s = tensor(ket1, plus, plus)
print("สถานะ:", np.round(s.real, 3))
print()
for lab, p in zip(labels_for(3), probabilities(s)):
    if p > 1e-9:
        print("  |{}>  P = {:.3f}".format(lab, p))
print()
print("=> มีแค่ 4 สถานะที่เป็นไปได้ ทุกตัวขึ้นต้นด้วย 1 เพราะคิวบิตแรกแน่นอน")
plot_probs(s, "|1> (x) |+> (x) |+>"); plt.show()

---
# ตอนที่ 9 — CNOT และสถานะเบลล์ทั้งสี่

> สไลด์หน้า 13–16

**CNOT** คือประตูสองคิวบิต: ถ้าคิวบิตควบคุมเป็น 1 ก็พลิกคิวบิตเป้าหมาย

$$a' = a, \qquad b' = a \oplus b$$

เทียบกับ XOR คลาสสิกที่รับ 2 บิตคืน 1 บิต (ข้อมูลหายไปหนึ่งบิต → ย้อนไม่ได้)
CNOT **เก็บอินพุตไว้ครบ** จึงย้อนกลับได้ ซึ่งเป็นข้อบังคับของควอนตัม

In [ ]:
# ประตูหลายคิวบิตจากสไลด์หน้า 13 -- เขียนเป็นเมทริกซ์ตรง ๆ
CNOT = np.array([[1,0,0,0],
                 [0,1,0,0],
                 [0,0,0,1],
                 [0,0,1,0]], dtype=complex)
SWAP = np.array([[1,0,0,0],
                 [0,0,1,0],
                 [0,1,0,0],
                 [0,0,0,1]], dtype=complex)
CZ   = np.diag([1, 1, 1, -1]).astype(complex)
CS   = np.diag([1, 1, 1, 1j]).astype(complex)

for name, U in [("CNOT", CNOT), ("SWAP", SWAP), ("CZ", CZ), ("CS", CS)]:
    print("{:5s} unitary? {}".format(name, is_unitary(U)))

print()
print("CNOT ส่งฐานไหนไปไหน")
for k, lab in enumerate(labels_for(2)):
    out = CNOT @ basis_state(2, k)
    print("   |{}>  ->  |{}>".format(lab, labels_for(2)[int(np.argmax(np.abs(out)))]))
print()
print("ตารางค่าความจริงตรงกับสไลด์หน้า 14:  a'=a, b'=a XOR b")

In [ ]:
# เวอร์ชันเร็วสำหรับ n คิวบิต
def apply_cx(state, c, t, n):
    s = np.asarray(state, dtype=complex).copy()
    for i in range(2 ** n):
        if (i >> (n - 1 - c)) & 1 and not (i >> (n - 1 - t)) & 1:
            j = i | (1 << (n - 1 - t))
            s[i], s[j] = s[j], s[i]
    return s

def apply_ccx(state, c1, c2, t, n):
    s = np.asarray(state, dtype=complex).copy()
    for i in range(2 ** n):
        if ((i >> (n-1-c1)) & 1) and ((i >> (n-1-c2)) & 1) and not (i >> (n-1-t)) & 1:
            j = i | (1 << (n - 1 - t))
            s[i], s[j] = s[j], s[i]
    return s

def apply_cz(state, c, t, n):
    s = np.asarray(state, dtype=complex).copy()
    for i in range(2 ** n):
        if ((i >> (n-1-c)) & 1) and ((i >> (n-1-t)) & 1):
            s[i] = -s[i]
    return s

print("ตรวจกับเมทริกซ์ 4x4:", np.allclose(apply_cx(basis_state(2,2), 0, 1, 2), CNOT @ basis_state(2,2)))
print("Toffoli |110> -> ", labels_for(3)[int(np.argmax(np.abs(apply_ccx(basis_state(3,6),0,1,2,3))))])
print("Toffoli |101> -> ", labels_for(3)[int(np.argmax(np.abs(apply_ccx(basis_state(3,5),0,1,2,3))))], " (ไม่เปลี่ยน)")

In [ ]:
# สถานะเบลล์ทั้งสี่ (สไลด์หน้า 15-16)
# วงจรเดียวกันหมด: H บน q0 แล้ว CNOT -- ต่างแค่อินพุตคลาสสิก 2 บิต
def make_bell(q0, q1):
    s = basis_state(2, q0 * 2 + q1)
    s = apply_1q(s, H, 0, 2)
    s = apply_cx(s, 0, 1, 2)
    return s

BELL_NAMES = {(0,0): "beta_00  (Phi+)", (0,1): "beta_01  (Psi+)",
              (1,0): "beta_10  (Phi-)", (1,1): "beta_11  (Psi-)"}

fig, axes = plt.subplots(1, 4, figsize=(13, 2.8))
for ax, (q0, q1) in zip(axes, [(0,0),(0,1),(1,0),(1,1)]):
    s = make_bell(q0, q1)
    terms = " + ".join("{:+.3f}|{}>".format(s[k].real, labels_for(2)[k])
                       for k in range(4) if abs(s[k]) > 1e-9)
    print("input |{}{}>  ->  {:16s} = {}".format(q0, q1, BELL_NAMES[(q0,q1)], terms))
    plot_amplitudes(s, "input |{}{}>".format(q0, q1), ax=ax)
plt.tight_layout(); plt.show()

print()
print("ทั้งสี่ตั้งฉากกัน -> เป็นฐานตั้งฉากปกติของ C^4")
B = np.array([make_bell(0,0), make_bell(0,1), make_bell(1,0), make_bell(1,1)])
print(np.round(np.abs(B.conj() @ B.T), 6))

In [ ]:
# ━━━ แบบฝึกหัด 9.1 ━━━
# SWAP สร้างได้จาก CNOT สามตัวต่อกันสลับทิศ: CNOT(0,1) -> CNOT(1,0) -> CNOT(0,1)
# จงยืนยันด้วยโค้ดว่าจริง

In [ ]:
#@title ✅ เฉลย 9.1 { display-mode: "form" }
built = np.zeros((4, 4), dtype=complex)
for k in range(4):
    s = basis_state(2, k)
    s = apply_cx(s, 0, 1, 2)
    s = apply_cx(s, 1, 0, 2)
    s = apply_cx(s, 0, 1, 2)
    built[:, k] = s
print("สร้างจาก CNOT สามตัว =")
print(np.round(built.real, 3))
print("SWAP =")
print(np.round(SWAP.real, 3))
print("เท่ากันไหม:", np.allclose(built, SWAP))

---
# ตอนที่ 10 — การพัวพัน · วัดด้วยอันดับเมทริกซ์

> สไลด์หน้า 15

สถานะส่วนใหญ่ใน $\mathbb{C}^4$ **เขียนเป็น $|a\rangle \otimes |b\rangle$ ไม่ได้** เรียกว่า **พัวพัน**

มีวิธีทดสอบที่สวยมากและมาจากพีชคณิตเชิงเส้นตรง ๆ:
จับเวกเตอร์ 4 ช่องมา **พับเป็นเมทริกซ์ 2×2** แล้วหา **อันดับ (rank)**

- rank = 1 → แยกเป็นผลคูณได้ (ไม่พัวพัน)
- rank > 1 → พัวพัน

เหตุผล: เมทริกซ์ที่เขียนเป็น $\mathbf{a}\mathbf{b}^T$ ได้ ก็คือเมทริกซ์อันดับ 1 พอดี
จำนวนค่าเอกฐาน (singular value) ที่ไม่เป็นศูนย์เรียกว่า **อันดับชมิดท์**

In [ ]:
def schmidt(state, nA=1):
    """พับสถานะเป็นเมทริกซ์แล้วหาค่าเอกฐาน -- ใช้ตัดสินการพัวพัน"""
    n = nqubits(state)
    M = np.asarray(state, dtype=complex).reshape(2 ** nA, 2 ** (n - nA))
    sv = np.linalg.svd(M, compute_uv=False)
    rank = int(np.sum(sv > 1e-9))
    return rank, sv

tests = {
    "|00>"                : basis_state(2, 0),
    "|+> (x) |0>"         : tensor(plus, ket0),
    "|+> (x) |->"         : tensor(plus, minus),
    "beta_00 (Bell)"      : make_bell(0, 0),
    "beta_11 (Bell)"      : make_bell(1, 1),
}
for name, s in tests.items():
    r, sv = schmidt(s)
    print("{:20s} rank={}  singular values={}   -> {}".format(
        name, r, np.round(sv, 3), "พัวพัน" if r > 1 else "แยกได้"))

In [ ]:
# ทำไม Bell แยกไม่ได้ -- พิสูจน์ด้วยพีชคณิตง่าย ๆ
print("สมมติ  beta_00 = (a0|0>+a1|1>) (x) (b0|0>+b1|1>)")
print("จะได้  [a0b0, a0b1, a1b0, a1b1] = [1/sqrt2, 0, 0, 1/sqrt2]")
print()
print("  a0*b0 = 1/sqrt2  -> a0 != 0 และ b0 != 0")
print("  a1*b1 = 1/sqrt2  -> a1 != 0 และ b1 != 0")
print("  แต่ a0*b1 ต้อง = 0  ขัดแย้งกันเอง  => แยกไม่ได้")
print()
print("ตรวจด้วยตัวเลข: ดีเทอร์มิแนนต์ของเมทริกซ์ที่พับได้")
for name, s in [("product |+>(x)|0>", tensor(plus, ket0)), ("Bell beta_00", make_bell(0,0))]:
    M = s.reshape(2, 2)
    print("  {:20s} det = {:+.4f}   (ศูนย์ = แยกได้)".format(name, np.linalg.det(M).real))

In [ ]:
# ความสัมพันธ์ของผลวัด: นี่คือสิ่งที่ "เห็นได้" จริงในการทดลอง
def measure_pair(state, shots=2000, seed=0):
    return measure(state, shots, seed)

prod = tensor(plus, plus)          # ไม่พัวพัน แต่ก็ 50/50 ทั้งคู่
bell = make_bell(0, 0)             # พัวพัน

fig, axes = plt.subplots(1, 2, figsize=(9, 2.9))
for ax, (name, s) in zip(axes, [("product  |+>(x)|+>", prod), ("entangled  beta_00", bell)]):
    c = measure_pair(s, 2000, seed=3)
    plot_counts(c, name, ax=ax)
    same = c.get("00", 0) + c.get("11", 0)
    print("{:22s} ผลตรงกัน (00 หรือ 11) = {:.1%}".format(name, same / 2000))
plt.tight_layout(); plt.show()

print()
print("สองกรณีนี้ คิวบิตแต่ละตัวให้ 50/50 เหมือนกันเป๊ะถ้าดูตัวเดียว")
print("ความต่างอยู่ที่ *ความสัมพันธ์* ซึ่งเห็นได้ต่อเมื่อเอาผลสองฝั่งมาเทียบกัน")
print("=> นี่คือเหตุผลที่การพัวพันส่งข้อมูลเร็วกว่าแสงไม่ได้")

In [ ]:
# ━━━ แบบฝึกหัด 10.1 ━━━
# สถานะ (|00> + |01> + |10> + |11>)/2 พัวพันหรือไม่
# ตอบด้วยอันดับชมิดท์ และถ้าแยกได้ จงบอกว่าแยกเป็นอะไร

In [ ]:
#@title ✅ เฉลย 10.1 { display-mode: "form" }
s = np.ones(4, dtype=complex) / 2
r, sv = schmidt(s)
print("สถานะ:", np.round(s.real, 3))
print("อันดับชมิดท์ =", r, " singular values =", np.round(sv, 3))
print()
if r == 1:
    M = s.reshape(2, 2)
    Um, sv_, Vm = np.linalg.svd(M)        # ระวัง: อย่าตั้งชื่อตัวแปรว่า S จะไปทับประตู S!
    fa = Um[:, 0] * np.sqrt(sv_[0]); fb = Vm[0, :] * np.sqrt(sv_[0])
    print("แยกได้เป็น:", np.round(fa, 3), " (x) ", np.round(fb, 3))
    print("ซึ่งก็คือ |+> (x) |+>  :", np.allclose(np.abs(np.kron(fa, fb)), np.abs(s)))

---
# ตอนที่ 11 — เทเลพอร์ตสถานะควอนตัม

> สไลด์หน้า 17–19

**โจทย์:** อลิซมีคิวบิต $|\phi\rangle = \alpha|0\rangle + \beta|1\rangle$ ที่เธอเองก็ไม่รู้ว่าคืออะไร
เธอส่งได้แค่ **บิตคลาสสิกสองตัว** ไปหาบ็อบ ทำยังไงให้บ็อบได้ $|\phi\rangle$

**วิธี:** ทั้งคู่ต้องแบ่งสถานะเบลล์กันไว้ก่อน แล้ว
1. อลิซทำ CNOT แล้ว H บนคิวบิตของเธอสองตัว (= วัดในฐานเบลล์)
2. อลิซวัดได้ผล $(i, j)$ ส่งไปให้บ็อบ
3. บ็อบแก้ด้วย $Z^i X^j$ ตามตารางในสไลด์หน้า 19

ผลคือบ็อบได้ $|\phi\rangle$ **เป๊ะ 100%** โดยไม่มีใครรู้เลยว่า $\alpha, \beta$ คืออะไร

In [ ]:
def measure_qubits(state, qubits, n, rng):
    """วัดเฉพาะคิวบิตที่ระบุ คืน (บิตที่วัดได้, สถานะหลังยุบ)"""
    state = np.asarray(state, dtype=complex)
    groups = {}
    for i in range(2 ** n):
        key = "".join(str((i >> (n - 1 - q)) & 1) for q in qubits)
        groups[key] = groups.get(key, 0.0) + abs(state[i]) ** 2
    keys = list(groups)
    ps = np.array([groups[k] for k in keys]); ps = ps / ps.sum()
    pick = keys[rng.choice(len(keys), p=ps)]
    out = state.copy()
    for i in range(2 ** n):
        key = "".join(str((i >> (n - 1 - q)) & 1) for q in qubits)
        if key != pick:
            out[i] = 0
    return pick, out / np.linalg.norm(out)

def teleport(phi, seed=None, verbose=True):
    """
    คิวบิต 0 = ข้อความลับของอลิซ
    คิวบิต 1 = ครึ่งของอลิซ        } คู่พัวพัน
    คิวบิต 2 = ครึ่งของบ็อบ        }
    """
    rng = np.random.default_rng(seed)
    n = 3
    # 1. เตรียม |phi> (x) |00>
    s = tensor(phi, ket0, ket0)
    # 2. สร้างคู่พัวพันระหว่างคิวบิต 1 กับ 2
    s = apply_1q(s, H, 1, n)
    s = apply_cx(s, 1, 2, n)
    # 3. อลิซวัดในฐานเบลล์ = CNOT(0->1) แล้ว H(0) แล้ววัดในฐานคำนวณ
    s = apply_cx(s, 0, 1, n)
    s = apply_1q(s, H, 0, n)
    bits, s = measure_qubits(s, [0, 1], n, rng)
    i, j = int(bits[0]), int(bits[1])
    # 4. ดึงสถานะของบ็อบออกมา (ตอนนี้ระบบแยกเป็นผลคูณแล้ว)
    idx0 = int(bits + "0", 2); idx1 = int(bits + "1", 2)
    bob = np.array([s[idx0], s[idx1]])
    bob = bob / np.linalg.norm(bob)
    # 5. บ็อบแก้ตามรหัสที่ได้รับ
    if j: bob = X @ bob
    if i: bob = Z @ bob
    fid = abs(np.vdot(phi, bob)) ** 2
    if verbose:
        corr = {(0,0): "I (ไม่ทำอะไร)", (0,1): "X", (1,0): "Z", (1,1): "Z X"}[(i,j)]
        print("  อลิซวัดได้ ({}, {})  ->  บ็อบใช้ {:14s} fidelity = {:.6f}".format(i, j, corr, fid))
    return bits, bob, fid

phi = qubit(1.1, 0.7)          # สถานะลับมั่ว ๆ
print("สถานะลับของอลิซ:", np.round(phi, 4))
print()
for k in range(6):
    teleport(phi, seed=k)

In [ ]:
# ตรวจครบทั้ง 4 ผลลัพธ์ และดูว่าแต่ละแบบออกบ่อยแค่ไหน
phi = qubit(2.0, 1.3)
rng_seeds = range(400)
tally, worst = {}, 1.0
for sd in rng_seeds:
    bits, bob, fid = teleport(phi, seed=sd, verbose=False)
    tally[bits] = tally.get(bits, 0) + 1
    worst = min(worst, fid)

print("สถานะลับ:", np.round(phi, 4))
print("ผลวัดของอลิซจาก 400 ครั้ง:", dict(sorted(tally.items())))
print("fidelity ต่ำสุดที่เจอ     :", round(worst, 12), " <- ได้ 1 ทุกครั้ง")
print()
print("สังเกตสองอย่าง:")
print("  1) ผลวัดออกมา 25% เท่ากันทุกแบบ และไม่มีข้อมูลเรื่อง phi เลย")
print("  2) บ็อบได้ phi กลับคืนครบทุกครั้ง หลังรู้ 2 บิตนั้น")
plot_counts(dict(sorted(tally.items())), "Alice's Bell-measurement outcomes (400 runs)"); plt.show()

In [ ]:
# ━━━ แบบฝึกหัด 11.1 ━━━
# ถ้าบ็อบ "ลืมแก้" คือไม่ใส่ X / Z ตามรหัส เขาจะได้อะไร
# ลองแก้ฟังก์ชัน teleport ให้ข้ามขั้นแก้ไข แล้วดู fidelity

In [ ]:
#@title ✅ เฉลย 11.1 { display-mode: "form" }
def teleport_nofix(phi, seed=None):
    rng = np.random.default_rng(seed); n = 3
    s = tensor(phi, ket0, ket0)
    s = apply_1q(s, H, 1, n); s = apply_cx(s, 1, 2, n)
    s = apply_cx(s, 0, 1, n); s = apply_1q(s, H, 0, n)
    bits, s = measure_qubits(s, [0, 1], n, rng)
    bob = np.array([s[int(bits+"0", 2)], s[int(bits+"1", 2)]])
    bob = bob / np.linalg.norm(bob)
    return bits, abs(np.vdot(phi, bob)) ** 2

phi = qubit(1.1, 0.7)
res = {}
for sd in range(300):
    b, f = teleport_nofix(phi, seed=sd)
    res.setdefault(b, []).append(f)
for b in sorted(res):
    print("  ผลวัด {} : fidelity เฉลี่ย {:.4f}   (เกิด {} ครั้ง)".format(b, np.mean(res[b]), len(res[b])))
print()
print("มีแค่ผลวัด 00 เท่านั้นที่บังเอิญถูกอยู่แล้ว อีกสามแบบได้สถานะผิด")
print("ถ้าเฉลี่ยรวมทุกกรณี บ็อบจะได้สถานะที่ไร้ข้อมูล -- เหมือนสถานะผสมในตอนที่ 7")
print("=> 2 บิตคลาสสิกนั้นจำเป็นจริง ๆ ข้อมูลไม่ได้วิ่งเร็วกว่าแสง")

---
# ตอนที่ 12 — Superdense coding

> สไลด์หน้า 20–27

กลับทิศจากเทเลพอร์ต: อยากส่ง **2 บิตคลาสสิก** แต่ส่งได้แค่ **1 คิวบิต**

| | ส่งอะไร | ใช้อะไร |
|---|---|---|
| เทเลพอร์ต | 1 คิวบิต | 2 บิตคลาสสิก + 1 คู่พัวพัน |
| Superdense | 2 บิตคลาสสิก | 1 คิวบิต + 1 คู่พัวพัน |

สี่ขั้นตามสไลด์: เตรียม → เข้ารหัส → ส่ง → ถอดรหัส

In [ ]:
def superdense(msg, verbose=True):
    n = 2
    b0, b1 = int(msg[0]), int(msg[1])
    stages = []
    # ขั้น 1 เตรียมคู่พัวพัน
    s = zero_state(n); s = apply_1q(s, H, 0, n); s = apply_cx(s, 0, 1, n)
    stages.append(("1 prepared", s.copy()))
    # ขั้น 2 A เข้ารหัสโดยแตะเฉพาะคิวบิตของตัวเอง (q0)
    if b1: s = apply_1q(s, X, 0, n)
    if b0: s = apply_1q(s, Z, 0, n)
    stages.append(("2 encoded", s.copy()))
    # ขั้น 3 ส่ง q0 ไปให้ B (คณิตศาสตร์ไม่เปลี่ยน เปลี่ยนแค่ว่าใครถือ)
    stages.append(("3 transmitted", s.copy()))
    # ขั้น 4 B ถอดรหัส
    s = apply_cx(s, 0, 1, n); s = apply_1q(s, H, 0, n)
    stages.append(("4 decoded", s.copy()))
    if verbose:
        enc = "I" if (not b0 and not b1) else ("Z X" if (b0 and b1) else ("X" if b1 else "Z"))
        print("ข้อความ '{}'  ->  A ใส่ประตู {}".format(msg, enc))
        got = labels_for(2)[int(np.argmax(probabilities(s)))]
        print("   B วัดได้ '{}'  ความน่าจะเป็น {:.4f}".format(got, probabilities(s).max()))
    return stages

fig, axes = plt.subplots(4, 4, figsize=(12, 9))
for row, msg in enumerate(["00", "01", "10", "11"]):
    stages = superdense(msg)
    for col, (name, s) in enumerate(stages):
        plot_amplitudes(s, "msg {} · {}".format(msg, name), ax=axes[row][col])
plt.tight_layout(); plt.show()
print()
print("สังเกตแถวที่ 2 ของแต่ละแถว: ความน่าจะเป็นไม่เปลี่ยนเลยตอนเข้ารหัส")
print("ข้อมูลไปซ่อนอยู่ในเครื่องหมายกับว่าคู่ไหนโผล่ ซึ่งดูจากคิวบิตเดียวไม่มีทางเห็น")

---
# ตอนที่ 13 — วงจรบวกเลข (half adder)

> สไลด์หน้า 28–31

เครื่องควอนตัมคำนวณได้ทุกอย่างที่เครื่องคลาสสิกทำได้ ตัวอย่างง่ายสุดคือ half adder

- **sum** = $a \oplus b$ → ใช้ CNOT สองตัว
- **carry** = $a \cdot b$ → ใช้ **Toffoli** (AND แบบย้อนกลับได้)

ใช้ 4 คิวบิต: `q0 = a`, `q1 = b`, `q2 = sum`, `q3 = carry`

In [ ]:
def half_adder(a, b, show=True):
    n = 4
    s = basis_state(n, (a << 3) | (b << 2))     # |a b 0 0>
    s = apply_cx(s, 0, 2, n)                    # sum ^= a
    s = apply_cx(s, 1, 2, n)                    # sum ^= b
    s = apply_ccx(s, 0, 1, 3, n)                # carry ^= a AND b
    idx = int(np.argmax(np.abs(s)))
    bits = format(idx, "04b")
    summ, carry = int(bits[2]), int(bits[3])
    if show:
        print("  {} + {} = {}{}_2 = {}     สถานะสุดท้าย |{}>".format(
            a, b, carry, summ, carry * 2 + summ, bits))
    return summ, carry

print("ตารางผลลัพธ์ (ตรงกับสไลด์หน้า 28)")
for a in [0, 1]:
    for b in [0, 1]:
        half_adder(a, b)

In [ ]:
# บวกทั้งสี่คู่พร้อมกันด้วยการซ้อนทับ -- แล้วดูว่าทำไมมันไม่ได้ช่วยอะไรเลย
n = 4
s = zero_state(n)
s = apply_1q(s, H, 0, n)        # a อยู่ในซ้อนทับ
s = apply_1q(s, H, 1, n)        # b อยู่ในซ้อนทับ
s = apply_cx(s, 0, 2, n); s = apply_cx(s, 1, 2, n); s = apply_ccx(s, 0, 1, 3, n)

print("หลังวงจร มีสี่สถานะที่เป็นไปได้ อย่างละ 25%:")
for k, p in enumerate(probabilities(s)):
    if p > 1e-9:
        bits = format(k, "04b")
        print("   |{}>  a={} b={} sum={} carry={}   P={:.2f}".format(
            bits, bits[0], bits[1], bits[2], bits[3], p))
print()
print("เครื่องคำนวณครบทั้งสี่คู่จริง ๆ แต่พอ 'วัด' เราได้คำตอบกลับมาแค่คู่เดียวแบบสุ่ม")
print("=> การซ้อนทับเฉย ๆ ไม่ได้ให้ความได้เปรียบ ต้องมีการแทรกสอดมาช่วยคัด (ตอนที่ 14-15)")
plot_probs(s, "half adder on all four inputs at once"); plt.show()

---
# ตอนที่ 14 — Deutsch–Jozsa

> สไลด์หน้า 29–34

มีฟังก์ชันลับ $f:\{0,1\}^n \to \{0,1\}$ รับประกันว่าเป็นแบบ **คงที่** (ตอบเหมือนกันหมด)
หรือ **สมดุล** (ตอบ 0 ครึ่ง 1 ครึ่ง) อย่างใดอย่างหนึ่ง

| | จำนวนครั้งที่ต้องถาม |
|---|---|
| คลาสสิก (แย่สุด) | $2^{n-1}+1$ |
| ควอนตัม | **1** |

**กลไก — phase kickback:** สไลด์เขียนออราเคิลเป็น $|x\rangle|y\rangle \to |x\rangle|y \oplus f(x)\rangle$
แต่ถ้าตั้งคิวบิตเสริมไว้ที่ $|-\rangle$ ผลจะกลายเป็น $(-1)^{f(x)}|x\rangle|-\rangle$
คือ **เครื่องหมายเด้งกลับมาติดที่รีจิสเตอร์อินพุตแทน**

เราจะเขียนทั้งสองแบบ แล้วตรวจว่าให้ผลเหมือนกัน

In [ ]:
def dj_phase_oracle(n, f):
    """แบบเฟส: |x> -> (-1)^f(x) |x>   (เข้าใจง่ายกว่า)"""
    def run():
        s = zero_state(n)
        for q in range(n): s = apply_1q(s, H, q, n)
        for x in range(2 ** n):
            if f(x): s[x] = -s[x]
        for q in range(n): s = apply_1q(s, H, q, n)
        return s
    return run()

def dj_xor_oracle(n, f):
    """แบบ XOR ตามสไลด์: ใช้คิวบิตเสริมตัวที่ n ตั้งไว้ที่ |->"""
    N = n + 1
    s = zero_state(N)
    s = apply_1q(s, X, n, N)                  # คิวบิตเสริม -> |1>
    for q in range(N): s = apply_1q(s, H, q, N)   # ทุกตัวรวมคิวบิตเสริม -> |->
    # ออราเคิล: พลิกคิวบิตเสริมเมื่อ f(x)=1
    out = s.copy()
    for i in range(2 ** N):
        x = i >> 1
        if f(x):
            out[i ^ 1] = s[i]
    s = out
    for q in range(n): s = apply_1q(s, H, q, N)
    # ความน่าจะเป็นของรีจิสเตอร์อินพุต (รวมคิวบิตเสริมออกไป)
    p = probabilities(s).reshape(2 ** n, 2).sum(axis=1)
    return p

FUNCS = {
    "constant 0" : lambda n: (lambda x: 0),
    "constant 1" : lambda n: (lambda x: 1),
    "balanced: MSB"   : lambda n: (lambda x: (x >> (n - 1)) & 1),
    "balanced: parity": lambda n: (lambda x: bin(x).count("1") % 2),
}

n = 3
print("n = {}  ({} อินพุต)  คลาสสิกแย่สุดต้องถาม {} ครั้ง".format(n, 2**n, 2**(n-1)+1))
print()
fig, axes = plt.subplots(1, 4, figsize=(13, 2.9))
for ax, (name, mk) in zip(axes, FUNCS.items()):
    f = mk(n)
    s = dj_phase_oracle(n, f)
    p_phase = probabilities(s)
    p_xor   = dj_xor_oracle(n, f)
    verdict = "คงที่" if p_phase[0] > 0.999 else "สมดุล"
    print("{:18s} P(000...0) = {:.4f}  ->  {:6s}   สองวิธีตรงกัน: {}".format(
        name, p_phase[0], verdict, np.allclose(p_phase, p_xor)))
    plot_probs(s, name, mark=0, ax=ax)
plt.tight_layout(); plt.show()
print()
print("อ่านผล: วัดได้ 0 ทั้งหมด -> คงที่   ·   วัดได้อย่างอื่น -> สมดุล   ถามออราเคิลครั้งเดียว")

In [ ]:
# ━━━ แบบฝึกหัด 14.1 ━━━
# สร้างฟังก์ชัน "สมดุล" แบบสุ่มของตัวเอง (ตอบ 1 ครึ่งหนึ่งของอินพุต)
# แล้วตรวจว่า Deutsch-Jozsa จับได้ว่าสมดุลจริง ลอง 20 ฟังก์ชันสุ่ม

In [ ]:
#@title ✅ เฉลย 14.1 { display-mode: "form" }
rng = np.random.default_rng(0)
n = 3; N = 2 ** n
ok = 0
for trial in range(20):
    idx = rng.permutation(N)
    table = np.zeros(N, dtype=int); table[idx[:N//2]] = 1   # ครึ่งหนึ่งตอบ 1
    f = lambda x, t=table: int(t[x])
    p0 = probabilities(dj_phase_oracle(n, f))[0]
    ok += (p0 < 1e-12)
print("ฟังก์ชันสมดุลสุ่ม 20 ตัว  ตรวจถูก {}/20".format(ok))
print()
print("และลองแบบคงที่บ้าง:")
for c in [0, 1]:
    p0 = probabilities(dj_phase_oracle(n, lambda x, c=c: c))[0]
    print("   f = {}   P(000) = {:.6f}  -> คงที่".format(c, p0))

---
# ตอนที่ 15 — อัลกอริทึมของโกรเวอร์

> สไลด์หน้า 46–51

หาของในกองที่ไม่เรียงลำดับ $N$ ชิ้น

| | จำนวนครั้งที่ต้องเปิดดู |
|---|---|
| คลาสสิก | $\sim N/2$ |
| โกรเวอร์ | $\sim \sqrt{N}$ |

**กลไก — amplitude amplification** ทำซ้ำสองขั้น:
1. **ออราเคิล** พลิกเครื่องหมายของคำตอบ
2. **สะท้อนรอบค่าเฉลี่ย** : $l_i^{new} = 2m - l_i$ (สูตรในสไลด์หน้า 47)

จำนวนรอบที่ดีที่สุดคือ $\left\lfloor \frac{\pi}{4}\sqrt{N/t} \right\rfloor$
**ทำเกินแล้วแย่ลง** เพราะแต่ละรอบคือการหมุนไปเป็นมุมคงที่ หมุนเลยก็เลยเป้า

In [ ]:
def grover_oracle(amp, marked):
    a = amp.copy()
    for m in marked: a[m] = -a[m]
    return a

def grover_diffuse(amp):
    """สะท้อนรอบค่าเฉลี่ย -- ตรงกับสูตร 2m - l_i ในสไลด์"""
    m = amp.mean()
    return 2 * m - amp

def grover_optimal(N, t):
    return max(1, int(np.floor(np.pi / 4 * np.sqrt(N / t))))

def grover(n, marked, iterations=None, trace=False):
    N = 2 ** n
    if iterations is None:
        iterations = grover_optimal(N, len(marked))
    amp = np.full(N, 1 / np.sqrt(N))
    hist = [amp.copy()]
    for _ in range(iterations):
        amp = grover_oracle(amp, marked)
        if trace: hist.append(amp.copy())
        amp = grover_diffuse(amp)
        hist.append(amp.copy())
    return amp, hist

n, marked = 4, [9]
N = 2 ** n
opt = grover_optimal(N, len(marked))
print("N = {}  คำตอบอยู่ที่ |{}>  รอบที่ดีที่สุด = {}".format(N, format(marked[0], "04b"), opt))
print()
amp = np.full(N, 1/np.sqrt(N))
print("เริ่มต้น: P(เจอ) = {:.4f}  (เดามั่ว = 1/{} )".format(amp[marked[0]]**2, N))
for k in range(1, opt + 3):
    a, _ = grover(n, marked, iterations=k)
    star = "  <- จุดที่ดีที่สุด" if k == opt else ("  (เลยไปแล้ว)" if k > opt else "")
    print("  รอบที่ {}: P(เจอ) = {:.4f}{}".format(k, a[marked[0]]**2, star))

In [ ]:
# ดูทีละขั้นว่าแท่งโตขึ้นยังไง
n, marked = 3, [5]
_, hist = grover(n, marked, iterations=2, trace=True)
titles = ["start (uniform)", "after oracle", "after diffusion", "after oracle", "after diffusion"]
fig, axes = plt.subplots(1, len(hist), figsize=(3.0 * len(hist), 2.9))
for ax, h, t in zip(axes, hist, titles):
    plot_amplitudes(h, t, mark=marked, ax=ax)
    ax.axhline(h.mean(), color=MARK, ls="--", lw=1.4)
plt.tight_layout(); plt.show()
print("เส้นประม่วง = ค่าเฉลี่ย  ·  แท่งม่วง = คำตอบที่ตามหา")
print()
print("กลไก: ออราเคิลทำให้คำตอบติดลบ -> ค่าเฉลี่ยถูกดึงลงนิดเดียว")
print("      แล้วการสะท้อนรอบค่าเฉลี่ยดีดคำตอบขึ้นไปสูงกว่าเดิม ส่วนตัวอื่นเตี้ยลง")

In [ ]:
# กราฟที่ตอบคำถาม "ทำไมทำเกินแล้วแย่ลง"
fig, ax = plt.subplots(figsize=(6.4, 3.2))
for n, t, col in [(4, 1, POS), (6, 1, NEG), (8, 1, MARK)]:
    N = 2 ** n
    ks = range(0, int(2.6 * np.sqrt(N)) + 1)
    ps = [grover(n, [1], iterations=k)[0][1] ** 2 for k in ks]
    ax.plot(list(ks), ps, color=col, lw=2.2, label="N = {}".format(N))
    o = grover_optimal(N, t)
    ax.plot([o], [grover(n, [1], iterations=o)[0][1] ** 2], "o", color=col, ms=7)
ax.axhline(1.0, color=MUTED, ls=":", lw=1)
ax.set_xlabel("Grover iterations"); ax.set_ylabel("P(success)")
ax.set_ylim(0, 1.08); ax.legend(frameon=False, fontsize=9)
ax.yaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.set_title("dots = floor(pi/4 * sqrt(N)) , the optimal stopping point")
plt.tight_layout(); plt.show()

print("เส้นโค้งขึ้นแล้วตกเป็นคาบ เพราะทุกรอบคือการหมุนเป็นมุมคงที่ในระนาบ 2 มิติ")
print("(ระนาบที่ขึงด้วยเวกเตอร์ 'คำตอบ' กับ 'ไม่ใช่คำตอบ')")

In [ ]:
# หลายคำตอบ และการวัดจริง
for n, marked in [(4, [3, 11]), (6, [20]), (8, [100])]:
    N = 2 ** n; t = len(marked)
    opt = grover_optimal(N, t)
    amp, _ = grover(n, marked)
    psucc = sum(amp[m] ** 2 for m in marked)
    print("N={:4d}  t={}  รอบ={:2d}  P(เจอ)={:.4f}   คลาสสิกต้องเปิดเฉลี่ย ~{} ครั้ง".format(
        N, t, opt, psucc, N // (t + 1)))

print()
# ยิงจริง
n, marked = 6, [20]
amp, _ = grover(n, marked)
c = measure(amp.astype(complex), shots=500, seed=2)
top = sorted(c.items(), key=lambda kv: -kv[1])[:4]
print("ยิง 500 ครั้งกับ N=64 ที่คำตอบคือ |{}>".format(format(20, "06b")))
for k, v in top:
    print("   |{}> : {} ครั้ง ({:.1%})".format(k, v, v/500))

In [ ]:
# ━━━ แบบฝึกหัด 15.1 ━━━
# N = 1024 (n=10) มีคำตอบ 1 ตัว
# 1) รอบที่ดีที่สุดคือเท่าไร  2) P(เจอ) เป็นเท่าไร
# 3) เทียบกับคลาสสิกที่ต้องเปิดเฉลี่ย 512 ครั้ง ประหยัดไปกี่เท่า

In [ ]:
#@title ✅ เฉลย 15.1 { display-mode: "form" }
n = 10; N = 2 ** n; marked = [777]
opt = grover_optimal(N, 1)
amp, _ = grover(n, marked)
print("N = {}".format(N))
print("รอบที่ดีที่สุด = {}   (sqrt(N) = {:.1f})".format(opt, np.sqrt(N)))
print("P(เจอ)       = {:.6f}".format(amp[marked[0]] ** 2))
print("คลาสสิกเฉลี่ย  = {} ครั้ง".format(N // 2))
print("ประหยัด       = {:.1f} เท่า".format((N // 2) / opt))
print()
print("หมายเหตุ: ความได้เปรียบเป็นแบบ 'ยกกำลังสอง' (quadratic) ไม่ใช่ 'เอกซ์โพเนนเชียล'")
print("         ต่างจากชอร์ที่เป็นเอกซ์โพเนนเชียล -- โกรเวอร์จึงไม่ทำให้ AES พังทันที")

---
# ตอนที่ 16 — Quantum Fourier Transform

> สไลด์หน้า 35–36

QFT เปลี่ยนจาก **ฐานคำนวณ** (เลขเขียนเป็นบิต 0/1) เป็น **ฐานฟูเรียร์** (เลขเขียนเป็นมุมเฟส)

$$\mathrm{QFT}|j\rangle = \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} e^{2\pi i jk/N}|k\rangle
= \frac{1}{\sqrt N}\bigotimes_{l=1}^{n}\Big(|0\rangle + e^{2\pi i j/2^{l}}|1\rangle\Big)$$

รูปขวามือบอกว่า หลัง QFT คิวบิตทุกตัวอยู่บนเส้นศูนย์สูตรหมด ต่างกันแค่ **มุม**
และมุมของแต่ละตัวหมุนด้วยความเร็วต่างกันเหมือนเข็มชั่วโมง/นาที/วินาที

In [ ]:
def qft_matrix(n):
    N = 2 ** n
    j = np.arange(N)
    return np.exp(2j * np.pi * np.outer(j, j) / N) / np.sqrt(N)

n = 3
F = qft_matrix(n)
print("QFT 3 คิวบิต เป็นยูนิแทรีไหม:", is_unitary(F))
print()
print("QFT|0> = การซ้อนทับเท่า ๆ กัน (เหมือน H ทุกตัว):")
print(np.round(F @ basis_state(n, 0), 3))
print()
print("QFT|1> = ซ้อนทับเหมือนกัน แต่เฟสไล่ขึ้นทีละขั้น:")
out = F @ basis_state(n, 1)
for k in range(2 ** n):
    print("   |{}>  ขนาด {:.3f}  เฟส {:+7.1f} องศา".format(
        labels_for(n)[k], abs(out[k]), np.degrees(np.angle(out[k]))))

In [ ]:
# ภาพ "นาฬิกา": เข็มแต่ละอันหมุนคนละความเร็วเมื่อ j เพิ่มทีละ 1
n = 4
fig, axes = plt.subplots(4, n, figsize=(2.0 * n, 8.0),
                         subplot_kw=dict(aspect="equal"))
for row, j in enumerate([0, 1, 2, 5]):
    for col in range(n):
        k = n - col                      # ตัวส่วน 2^k : ซ้าย=ช้าสุด ขวา=เร็วสุด
        ang = 2 * np.pi * j / (2 ** k)
        ax = axes[row][col]
        th = np.linspace(0, 2*np.pi, 100)
        ax.plot(np.cos(th), np.sin(th), color=GRID, lw=1)
        ax.arrow(0, 0, np.cos(ang)*0.82, np.sin(ang)*0.82,
                 head_width=0.13, color=MARK, lw=2)
        ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.axis("off")
        if row == 0: ax.set_title("phase j/2^{}".format(k), fontsize=9)
        if col == 0: ax.text(-1.9, 0, "j = {}".format(j), fontsize=10, va="center", color=INK)
plt.suptitle("QFT: the same number j stored as angles", fontsize=11, y=0.995)
plt.tight_layout(); plt.show()

print("เข็มขวาสุด (ตัวส่วน 2^1) หมุนครึ่งรอบทุกครั้งที่ j เพิ่ม 1")
print("เข็มถัดไปทางซ้ายหมุนหนึ่งในสี่รอบ  ไล่ไปจนซ้ายสุดช้าที่สุด")
print("=> นี่คือเลขฐานสองที่เขียนด้วย 'มุม' แทน '0/1'")
print("   และเพราะมันเป็นมุม มันจึง 'รู้จักคาบ' ซึ่งเป็นหัวใจของตอนที่ 17-18")

In [ ]:
# QFT ดึงคาบออกมาได้ -- ตัวอย่างตรง ๆ
n = 6; N = 2 ** n
for r in [4, 8, 3]:
    v = np.zeros(N, dtype=complex)
    v[::r] = 1                       # สัญญาณที่มีคาบ r
    v /= np.linalg.norm(v)
    out = qft_matrix(n) @ v
    peaks = [k for k in range(N) if abs(out[k]) ** 2 > 0.5 * (np.abs(out) ** 2).max()]
    print("สัญญาณคาบ r={:2d}  ->  ยอดอยู่ที่ k = {}   (ห่างกัน {} = {}/{})".format(
        r, peaks, N // r if N % r == 0 else round(N / r, 1), N, r))

fig, axes = plt.subplots(1, 2, figsize=(10, 2.8))
v = np.zeros(N, dtype=complex); v[::8] = 1; v /= np.linalg.norm(v)
axes[0].bar(range(N), v.real, color=POS, width=0.8); axes[0].set_title("input: period 8 in x")
axes[1].bar(range(N), np.abs(qft_matrix(n) @ v) ** 2, color=MARK, width=0.8)
axes[1].set_title("after QFT: spikes every 64/8 = 8")
for ax in axes:
    ax.yaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

---
# ตอนที่ 17 — Quantum Phase Estimation

> สไลด์หน้า 37–40

ถ้ามียูนิแทรี $U$ กับไอเกนเวกเตอร์ $|u\rangle$ ที่ $U|u\rangle = e^{2\pi i\theta}|u\rangle$
**QPE หาค่า $\theta$ ให้เรา**

วิธี:
1. รีจิสเตอร์นับ $t$ คิวบิต → H ทุกตัว
2. ทำ controlled-$U^{2^k}$ → เฟสเด้งกลับมาเขียนลงรีจิสเตอร์นับ (phase kickback)
3. **QFT ผกผัน** → แปลงจากฐานฟูเรียร์กลับเป็นเลขฐานสองที่อ่านได้
4. วัด → ได้ $y$ แล้ว $\theta \approx y/2^t$

นี่คือจุดที่ **ไอเกนแวลู (ตอนที่ 5 ของหน้าเว็บ)** มาเจอกับ **QFT (ตอนที่ 16)**

In [ ]:
def qpe(theta, t, verbose=True):
    """
    U = เกตเฟส P(2*pi*theta) ซึ่งมี |1> เป็นไอเกนเวกเตอร์ ไอเกนแวลู e^(2*pi*i*theta)
    คิวบิต 0..t-1 = รีจิสเตอร์นับ  ·  คิวบิต t = ไอเกนสเตต ตั้งไว้ที่ |1>
    """
    n = t + 1
    s = zero_state(n)
    s = apply_1q(s, X, t, n)                  # ไอเกนสเตต |1>
    for q in range(t):
        s = apply_1q(s, H, q, n)
    # controlled-U^(2^(t-1-q)) : ใส่เฟสเมื่อทั้งคิวบิตคุมและไอเกนสเตตเป็น 1
    for q in range(t):
        ang = 2 * np.pi * theta * (2 ** (t - 1 - q))
        for i in range(2 ** n):
            if ((i >> (n - 1 - q)) & 1) and ((i >> (n - 1 - t)) & 1):
                s[i] *= np.exp(1j * ang)
    # QFT ผกผันบนรีจิสเตอร์นับ
    psi = s.reshape(2 ** t, 2)
    psi = qft_matrix(t).conj().T @ psi
    s = psi.reshape(-1)
    # ความน่าจะเป็นของรีจิสเตอร์นับ
    p = probabilities(s).reshape(2 ** t, 2).sum(axis=1)
    y = int(np.argmax(p))
    if verbose:
        print("theta จริง = {:.5f}   t = {} คิวบิต".format(theta, t))
        print("   อ่านได้ y = {} = {}_2   ->  theta ประมาณ = {}/{} = {:.5f}".format(
            y, format(y, "0{}b".format(t)), y, 2**t, y / 2**t))
        print("   P ของยอดสูงสุด = {:.4f}   คลาดเคลื่อน = {:.5f}".format(p[y], abs(y/2**t - theta)))
    return p

print("--- กรณีที่ theta เขียนด้วย t บิตได้ลงตัว ---")
p1 = qpe(0.375, 3)        # 3/8 = 0.011 ฐานสอง
print()
print("--- กรณีที่เขียนไม่ลงตัว ---")
p2 = qpe(1/3, 3)
print()
p3 = qpe(1/3, 6)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 2.9))
for ax, (p, ttl) in zip(axes, [(p1, "theta=3/8, t=3  (exact)"),
                               (p2, "theta=1/3, t=3  (spread)"),
                               (p3, "theta=1/3, t=6  (sharper)")]):
    ax.bar(range(len(p)), p, color=[MARK if i == int(np.argmax(p)) else POS for i in range(len(p))], width=0.7)
    ax.set_title(ttl); ax.set_ylim(0, 1.05); ax.set_ylabel("P(y)")
    ax.yaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print("เพิ่มคิวบิตนับ 1 ตัว -> ความละเอียดดีขึ้นเท่าตัว นี่คือเหตุผลที่ชอร์ต้องใช้คิวบิตเยอะ")

In [ ]:
# ━━━ แบบฝึกหัด 17.1 ━━━
# ประตู S มีไอเกนแวลู e^(2*pi*i*theta) สำหรับไอเกนเวกเตอร์ |1> -- theta เท่าไร
# ใช้ qpe() ตรวจคำตอบ

In [ ]:
#@title ✅ เฉลย 17.1 { display-mode: "form" }
print("S = diag(1, i) ดังนั้น S|1> = i|1>")
print("i = e^(i*pi/2) = e^(2*pi*i*(1/4))   =>  theta = 1/4")
print()
vals = np.linalg.eigvals(S)
print("ตรวจด้วยไอเกนแวลูจาก numpy:", np.round(vals, 6))
print()
p = qpe(0.25, 3)
print()
print("t=3 อ่านได้ y=2 -> theta = 2/8 = 0.25 ถูกต้อง และแม่น 100% เพราะ 1/4 = 0.010 ฐานสองพอดี")

---
# ตอนที่ 18 — อัลกอริทึมของชอร์

> สไลด์หน้า 41–45

ชอร์ **ไม่ได้แยกตัวประกอบตรง ๆ** แต่เปลี่ยนโจทย์เป็นการ **หาคาบ** ของ $f(x) = a^x \bmod N$

**ส่วนคลาสสิก** (ทำบนคอมพิวเตอร์ธรรมดาได้ทั้งหมด):
1. สุ่ม $a$ จาก 2 ถึง $N-1$
2. ถ้า $\gcd(a,N) \neq 1$ → โชคดี ได้ตัวประกอบเลย จบ
3. ไม่งั้น หาคาบ $r$ (← **ขั้นนี้เท่านั้นที่ใช้ควอนตัม**)
4. ถ้า $r$ เป็นคี่ หรือ $a^{r/2} \equiv -1$ → สุ่ม $a$ ใหม่
5. ไม่งั้น ตัวประกอบคือ $\gcd(a^{r/2} \pm 1,\ N)$

**ส่วนควอนตัม** คือ QFT อ่านคาบจากสเปกตรัม — เหมือนตอนที่ 16 เป๊ะ

> สังเกตตอนรัน: บางครั้งขั้นควอนตัมคืนค่า $r$ ที่ผิด (เช่นวัดได้ $y=0$ พอดี)
> อัลกอริทึมจริงก็เป็นแบบนี้ — มันเป็นอัลกอริทึมแบบ **น่าจะเป็น** เราจึงต้อง
> *ตรวจคำตอบด้วยการคำนวณคลาสสิก* ทุกครั้ง แล้วลองใหม่ถ้าไม่ผ่าน

In [ ]:
from math import gcd
from fractions import Fraction

def classical_period(a, N):
    """หาคาบแบบคลาสสิก (ไล่ดูทีละตัว) -- ใช้เทียบคำตอบเท่านั้น"""
    if gcd(a, N) != 1: return None
    r = 1
    while pow(a, r, N) != 1:
        r += 1
        if r > N: return None
    return r

N, a = 35, 4                      # ตัวอย่างในสไลด์หน้า 41
print("f(x) = {}^x mod {}".format(a, N))
print("  x   :", "  ".join("{:3d}".format(x) for x in range(12)))
print("  f(x):", "  ".join("{:3d}".format(pow(a, x, N)) for x in range(12)))
print()
print("เห็นลายซ้ำไหม -> คาบ r =", classical_period(a, N))

In [ ]:
def quantum_period(a, N, t=10, seed=0, show=True):
    """
    จำลองขั้นควอนตัม:
      1) รีจิสเตอร์ 1 = ซ้อนทับ x ทั้งหมด, รีจิสเตอร์ 2 เก็บ f(x)
      2) วัดรีจิสเตอร์ 2 -> รีจิสเตอร์ 1 ยุบเหลือ x ที่ให้ค่าเดียวกัน (ห่างกันทีละ r)
      3) ทำ QFT บนรีจิสเตอร์ 1 แล้ววัด -> ได้ y ที่ y/2^t ใกล้ k/r
      4) ใช้เศษส่วนต่อเนื่อง (continued fractions) ดึง r ออกมา
    """
    rng = np.random.default_rng(seed)
    L = 2 ** t
    f = np.array([pow(a, x, N) for x in range(L)])
    # ขั้น 2 : วัดรีจิสเตอร์ 2
    x0 = int(rng.integers(L))
    keep = np.where(f == f[x0])[0]
    state = np.zeros(L, dtype=complex)
    state[keep] = 1 / np.sqrt(len(keep))
    # ขั้น 3 : QFT (ใช้ FFT เพื่อความเร็ว -- เทียบเท่า qft_matrix ทุกประการ)
    out = np.fft.ifft(state) * np.sqrt(L)
    p = np.abs(out) ** 2
    y = int(rng.choice(L, p=p / p.sum()))
    # ขั้น 4 : เศษส่วนต่อเนื่อง
    r = Fraction(y, L).limit_denominator(N).denominator
    if show:
        print("   วัดรีจิสเตอร์ 2 ได้ f = {}  -> x ที่เหลือคือ {} ... (ห่างกันทีละคาบ)".format(
            int(f[x0]), [int(v) for v in keep[:5]]))
        print("   วัดหลัง QFT ได้ y = {}   ->  y/2^t = {}/{} = {:.5f}".format(y, y, L, y / L))
        print("   เศษส่วนต่อเนื่อง -> r = {}".format(r))
    return r, p

print("หาคาบของ 4^x mod 35 ด้วยควอนตัม")
r, p = quantum_period(4, 35, t=10, seed=3)
print("   ตรวจ: 4^{} mod 35 = {}  (ต้องได้ 1)".format(r, pow(4, r, 35)))
print()

L = 2 ** 10
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.bar(range(L), p, color=POS, width=1.4)
true_r = classical_period(4, 35)
for k in range(true_r):
    ax.axvline(k * L / true_r, color=MARK, ls="--", lw=1.1)
ax.set_xlabel("measured value y"); ax.set_ylabel("probability")
ax.set_title("QFT spectrum: spikes at multiples of 1024/{} = {:.1f}".format(true_r, L/true_r))
ax.yaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print("เส้นประม่วง = ตำแหน่งที่ทฤษฎีบอกว่ายอดต้องอยู่")

In [ ]:
def shor(N, t=10, max_tries=25, seed=0, verbose=True):
    """อัลกอริทึมชอร์เต็มรูปแบบ (ส่วนควอนตัมถูกจำลอง)"""
    rng = np.random.default_rng(seed)
    if N % 2 == 0:
        return (2, N // 2)
    for attempt in range(max_tries):
        a = int(rng.integers(2, N))
        g = gcd(a, N)
        if g > 1:
            if verbose: print("  a={:2d} : gcd(a,N)={} != 1  -> โชคดี ได้ตัวประกอบเลย".format(a, g))
            return (g, N // g)
        r, _ = quantum_period(a, N, t=t, seed=int(rng.integers(1e6)), show=False)
        if r == 0 or pow(a, r, N) != 1:
            if verbose: print("  a={:2d} : r={} ตรวจไม่ผ่าน (a^r mod N != 1) -> สุ่มใหม่".format(a, r))
            continue
        if r % 2 == 1:
            if verbose: print("  a={:2d} : r={} เป็นเลขคี่ -> สุ่มใหม่".format(a, r))
            continue
        half = pow(a, r // 2, N)
        if half == N - 1:
            if verbose: print("  a={:2d} : r={}, a^(r/2) = -1 mod N -> สุ่มใหม่".format(a, r))
            continue
        f1, f2 = gcd(half - 1, N), gcd(half + 1, N)
        if 1 < f1 < N:
            if verbose: print("  a={:2d} : r={}, a^(r/2)={}  ->  gcd ให้ {} และ {}".format(a, r, half, f1, f2))
            return (f1, N // f1)
        if 1 < f2 < N:
            if verbose: print("  a={:2d} : r={}, a^(r/2)={}  ->  gcd ให้ {} และ {}".format(a, r, half, f1, f2))
            return (f2, N // f2)
        if verbose: print("  a={:2d} : r={} ได้แค่ 1 กับ N -> สุ่มใหม่".format(a, r))
    return None

for N in [15, 21, 35, 33]:
    print("แยกตัวประกอบ N = {}".format(N))
    res = shor(N, t=10, seed=N)
    print("  => {} = {} x {}".format(N, res[0], res[1]))
    print()

In [ ]:
# ━━━ แบบฝึกหัด 18.1 ━━━
# ตามสไลด์หน้า 41: N=35, a=4 ได้คาบ r=6
# 1) ตรวจว่า r=6 จริงไหม
# 2) a^(r/2) mod N เป็นเท่าไร และเงื่อนไข a^(r/2) != -1 mod N ผ่านไหม
# 3) หาตัวประกอบด้วย gcd

In [ ]:
#@title ✅ เฉลย 18.1 { display-mode: "form" }
N, a = 35, 4
r = classical_period(a, N)
print("1) คาบ r =", r, "  ตรวจ: {}^{} mod {} = {}".format(a, r, N, pow(a, r, N)))
half = pow(a, r // 2, N)
print("2) a^(r/2) mod N = {}^{} mod {} = {}".format(a, r//2, N, half))
print("   เงื่อนไข a^(r/2) != N-1 = {} ?  {}".format(N - 1, half != N - 1))
f1, f2 = gcd(half - 1, N), gcd(half + 1, N)
print("3) gcd({} - 1, {}) = {}".format(half, N, f1))
print("   gcd({} + 1, {}) = {}".format(half, N, f2))
print("   => {} = {} x {}".format(N, f1, f2))

---
# ตอนที่ 19 — ทำซ้ำด้วย Qiskit ของจริง

> สไลด์หน้า 28–31

ทุกอย่างที่ผ่านมาเราเขียนเองด้วย numpy ตอนนี้ลองใช้ Qiskit ดูบ้าง
จะเห็นว่าโค้ดสั้นลงมาก แต่ **ข้างในก็ทำสิ่งเดียวกันกับที่เราเพิ่งเขียน**

เซลล์ถัดไปติดตั้ง Qiskit (ใช้เวลาราว 1 นาทีครั้งแรก) ถ้าติดตั้งไม่ได้
ส่วนที่เหลือของ notebook ก็ยังใช้ได้ปกติ เพราะไม่ได้พึ่ง Qiskit

In [ ]:
QISKIT = False
try:
    import qiskit
except ImportError:
    import subprocess, sys
    print("กำลังติดตั้ง Qiskit ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "qiskit", "qiskit-aer", "pylatexenc"], check=False)
try:
    from qiskit import QuantumCircuit, transpile
    from qiskit.quantum_info import Statevector
    from qiskit_aer import AerSimulator
    QISKIT = True
    print("Qiskit พร้อมใช้งาน เวอร์ชัน", qiskit.__version__)
except Exception as e:
    print("ใช้ Qiskit ไม่ได้ในสภาพแวดล้อมนี้:", e)
    print("ข้ามตอนที่ 19 ไปได้เลย ตอนอื่นไม่ได้พึ่ง Qiskit")

In [ ]:
if QISKIT:
    sim = AerSimulator()

    def run_qc(qc, shots=1024):
        return sim.run(transpile(qc, sim), shots=shots).result().get_counts()

    # --- สถานะเบลล์ ---
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    print(qc)
    counts = run_qc(qc, 2000)
    print("\nผลวัด:", counts)
    plot_counts(counts, "Qiskit: Bell state"); plt.show()

    # เทียบกับที่เราคำนวณเอง
    qc2 = QuantumCircuit(2); qc2.h(0); qc2.cx(0, 1)
    sv = np.asarray(Statevector.from_instruction(qc2))
    print("Qiskit statevector :", np.round(sv, 4))
    print("ของเราเอง          :", np.round(make_bell(0, 0), 4))
    print("ตรงกันไหม          :", np.allclose(np.abs(sv), np.abs(make_bell(0, 0))))
    print()
    print("หมายเหตุ: Qiskit เรียงบิตกลับด้าน (little-endian) ดังนั้นสำหรับสถานะที่ไม่สมมาตร")
    print("         ลำดับช่องอาจสลับกับของเรา -- เป็นเรื่องของ convention ไม่ใช่ฟิสิกส์")

In [ ]:
if QISKIT:
    # --- half adder ตามสไลด์หน้า 28-31 ---
    def qiskit_half_adder(a, b):
        qc = QuantumCircuit(4, 2)
        if a: qc.x(0)
        if b: qc.x(1)
        qc.barrier()
        qc.cx(0, 2)          # sum ^= a
        qc.cx(1, 2)          # sum ^= b
        qc.ccx(0, 1, 3)      # carry ^= a AND b   <- Toffoli
        qc.barrier()
        qc.measure(2, 0)     # sum
        qc.measure(3, 1)     # carry
        return qc

    qc = qiskit_half_adder(1, 1)
    print(qc)
    print()
    for a in [0, 1]:
        for b in [0, 1]:
            c = run_qc(qiskit_half_adder(a, b), 512)
            bits = max(c, key=c.get)          # 'carry sum' (Qiskit อ่านขวาไปซ้าย)
            carry, summ = int(bits[0]), int(bits[1])
            print("  {} + {} -> sum={} carry={}  = {}".format(a, b, summ, carry, carry*2+summ))

In [ ]:
if QISKIT:
    # --- เทเลพอร์ต (ใช้ประตูควบคุมแทนการควบคุมด้วยบิตคลาสสิก ผลเท่ากัน) ---
    theta, phi = 1.1, 0.7
    qc = QuantumCircuit(3)
    qc.ry(theta, 0); qc.rz(phi, 0)      # เตรียมสถานะลับบน q0
    qc.barrier()
    qc.h(1); qc.cx(1, 2)                # คู่พัวพัน q1-q2
    qc.barrier()
    qc.cx(0, 1); qc.h(0)                # อลิซวัดในฐานเบลล์
    qc.barrier()
    qc.cx(1, 2); qc.cz(0, 2)            # บ็อบแก้
    print(qc)

    from qiskit.quantum_info import partial_trace, state_fidelity
    sv = Statevector.from_instruction(qc)
    rho_bob = partial_trace(sv, [0, 1])          # ตัดคิวบิต 0,1 ทิ้ง เหลือของบ็อบ
    qt = QuantumCircuit(1); qt.ry(theta, 0); qt.rz(phi, 0)
    target = Statevector.from_instruction(qt)
    print("\nfidelity ของสถานะที่บ็อบได้ = {:.10f}".format(state_fidelity(rho_bob, target)))
    print("=> เทเลพอร์ตสำเร็จ 100% ตรงกับที่เราจำลองเองในตอนที่ 11")

In [ ]:
if QISKIT:
    # --- Grover ด้วย Qiskit : ค้นหา |101> ใน 3 คิวบิต ---
    def grover_qiskit(n=3, target="101", iterations=None):
        if iterations is None:
            iterations = grover_optimal(2 ** n, 1)
        qc = QuantumCircuit(n, n)
        qc.h(list(range(n)))
        for _ in range(iterations):
            # ออราเคิล: พลิกเฟสของสถานะเป้าหมาย
            for i, bit in enumerate(reversed(target)):
                if bit == "0": qc.x(i)
            qc.h(n - 1); qc.mcx(list(range(n - 1)), n - 1); qc.h(n - 1)
            for i, bit in enumerate(reversed(target)):
                if bit == "0": qc.x(i)
            # diffusion: สะท้อนรอบค่าเฉลี่ย
            qc.h(list(range(n))); qc.x(list(range(n)))
            qc.h(n - 1); qc.mcx(list(range(n - 1)), n - 1); qc.h(n - 1)
            qc.x(list(range(n))); qc.h(list(range(n)))
        qc.measure(list(range(n)), list(range(n)))
        return qc, iterations

    qc, it = grover_qiskit(3, "101")
    c = run_qc(qc, 2000)
    print("Grover n=3 หา |101> ด้วย {} รอบ".format(it))
    print("ผลวัด:", dict(sorted(c.items(), key=lambda kv: -kv[1])))
    print("โอกาสเจอเป้าหมาย: {:.1%}".format(c.get("101", 0) / 2000))
    plot_counts(c, "Qiskit Grover · target |101>"); plt.show()
    print()
    print("เทียบกับที่เราคำนวณเอง: {:.1%}".format(grover(3, [5])[0][5] ** 2))

---
# ตอนที่ 20 — สรุปและทางไปต่อ

## สิ่งที่ควรติดตัวไปจาก notebook นี้

**1. ควอนตัมคอมพิวติ้ง = พีชคณิตเชิงเส้น + กฎสุ่มหนึ่งข้อ**
ทุกเซลล์ที่รันไปเป็น `np.array`, `@`, และ `np.kron` ทั้งนั้น ไม่มีฟิสิกส์ลึกลับเลย
กฎเดียวที่ไม่ใช่พีชคณิตเชิงเส้นคือ **กฎของบอร์น** ในตอนที่ 3

**2. แอมพลิจูดไม่ใช่ความน่าจะเป็น**
แอมพลิจูดติดลบและเป็นจำนวนเชิงซ้อนได้ จึง **หักล้างกันได้** ความน่าจะเป็นทำไม่ได้
ความต่างข้อนี้ข้อเดียวคือที่มาของ Deutsch–Jozsa, Grover และ Shor ทั้งหมด

**3. การซ้อนทับเฉย ๆ ไม่ได้ช่วยอะไร**
ตอนที่ 13 แสดงให้เห็นชัด: คำนวณทั้งสี่คู่พร้อมกันได้ แต่วัดแล้วได้คำตอบเดียวแบบสุ่มอยู่ดี
อัลกอริทึมควอนตัมที่ดีคือการ **จัดเฟสให้คำตอบผิดหักล้างกันหมด**

**4. ทุกอย่างต้องย้อนกลับได้**
เพราะยูนิแทรีมีตัวผกผันเสมอ จึงไม่มีประตู AND/OR ต้องใช้ CNOT และ Toffoli แทน

## ตารางแปลศัพท์

| พีชคณิตเชิงเส้น | ควอนตัม | ในโค้ด |
|---|---|---|
| เวกเตอร์หนึ่งหน่วย | สถานะ $\lvert\psi\rangle$ | `np.array([...])` ที่ norm = 1 |
| ผลคูณภายใน | $\langle\phi\vert\psi\rangle$ | `np.vdot(phi, psi)` |
| ผลคูณภายนอก | $\rho = \lvert\psi\rangle\langle\psi\rvert$ | `np.outer(psi, psi.conj())` |
| เมทริกซ์มุมฉาก $Q^TQ=I$ | ยูนิแทรี $U^\dagger U=I$ | `U.conj().T @ U` |
| การคูณเมทริกซ์ | ประตูต่อกันในวงจร | `G2 @ G1` |
| ผลคูณเทนเซอร์ | หลายคิวบิต | `np.kron(a, b)` |
| อันดับเมทริกซ์ = 1 | สถานะแยกได้ (ไม่พัวพัน) | `np.linalg.svd` |
| ไอเกนแวลู | ค่าที่วัดได้ / เฟสใน QPE | `np.linalg.eig` |
| เมทริกซ์ DFT | QFT | `np.fft` |

## ไปต่อยังไง

1. **ทบทวนพีชคณิตเชิงเส้นสองบท** — ฐานตั้งฉาก/Gram–Schmidt และไอเกน/ทฤษฎีบทสเปกตรัม
   สองเรื่องนี้ถูกใช้ซ้ำตลอดในควอนตัม
2. **ลองแก้โค้ดใน notebook นี้** — เปลี่ยน $n$, เปลี่ยนออราเคิล, ทำ Grover ที่มี 3 คำตอบ
   ถ้าแก้แล้วทำนายผลถูกก่อนรัน แปลว่าเข้าใจจริง
3. **ทำ Assignment II** — สไลด์ให้ลองไฟล์ `Grover's algorithm.ipynb`,
   `quantum_fourier_transform.ipynb`, `phase_estimation.ipynb`, `Shor's algorithm.ipynb`
   บน IBM Quantum Lab ตอนนี้คุณรู้แล้วว่าข้างในแต่ละไฟล์ทำอะไร
4. **รันบนเครื่องจริง** — สไลด์หน้า 28 พูดถึง error map กับการเลือก initial layout
   ลองรันวงจรเดิมบนฮาร์ดแวร์จริงแล้วเทียบกับผลจำลอง จะเห็นผลของ noise ชัดมาก

## หนังสือ

- **Nielsen & Chuang**, *Quantum Computation and Quantum Information* — เล่มมาตรฐานที่สไลด์อ้างถึง
- **Yanofsky & Mannucci**, *Quantum Computing for Computer Scientists* — สายคอมพิวเตอร์อ่านง่ายกว่า
- **Bruce Solomon**, *Linear Algebra, Geometry and Transformation* — ถ้าอยากปูพีชคณิตเชิงเส้นให้แน่นก่อน

---

*เนื้อหาเรียงตามสไลด์ Introduction to Quantum Computing โดย Kamonluk Suksen, Ph.D.*